In [2]:

import requests
from bs4 import BeautifulSoup

# 국토교통부 교통물류 보도자료 목록
url = "https://www.molit.go.kr/USR/NEWS/m_71/lst.jsp?search_section=p_sec_5"

response = requests.get(url, timeout=15)
response.raise_for_status()
response.encoding = response.apparent_encoding

soup = BeautifulSoup(response.text, "html.parser")

# 목록 페이지에 포함된 링크 확인
links = soup.select("a[href]")

print("HTTP 상태:", response.status_code)
print("전체 링크 수:", len(links))

print("\n===== 보도자료 관련 링크 후보 =====")

for link in links:
    href = link.get("href", "")
    title = link.get_text(" ", strip=True)

    if "dtl.jsp" in href and title:
        print("제목:", title[:100])
        print("링크:", href[:200])
        print()


HTTP 상태: 200
전체 링크 수: 144

===== 보도자료 관련 링크 후보 =====
제목: 물류기업과 기술기업이 만나는  ‘협업의 장’ 열린다
링크: dtl.jsp?lcmspage=1&id=95092592

제목: 김용석 대광위원장,고양창릉 광역교통개선대책 추진상황 점검
링크: dtl.jsp?lcmspage=1&id=95092593

제목: [차관동정] 정의경 제2차관, “수도권 서부지역 교통난 해소, 철도·도로사업 조기 완공에 속도 내야”
링크: dtl.jsp?lcmspage=1&id=95092583

제목: 신도시 교통대책 속도 높인다 … 도로사업 16곳 집중 점검
링크: dtl.jsp?lcmspage=1&id=95092576

제목: 전남광주 어르신, 카드 한 장으로 도시철도 무료·교통비 환급
링크: dtl.jsp?lcmspage=1&id=95092574

제목: 김용석 대광위원장,“하남교산 주요 교통시설 적기 구축에 총력”
링크: dtl.jsp?lcmspage=1&id=95092564

제목: 문체부-국토부, 관광교통 칸막이 허물고 ‘지방관광 시대’ 연다.
링크: dtl.jsp?lcmspage=1&id=95092547

제목: 3기 신도시 광역교통시설 적기 구축에 총력
링크: dtl.jsp?lcmspage=1&id=95092532

제목: 제34회 물류의 날 기념식 … 물류산업의 지속가능 발전방향 모색
링크: dtl.jsp?lcmspage=1&id=95092531

제목: 3기 신도시 교통대책  ‘직접 인·허가’로 속도 높인다
링크: dtl.jsp?lcmspage=1&id=95092519



In [4]:

import pandas as pd
from urllib.parse import urljoin

# 보도자료 제목과 상세 URL을 저장할 목록
news_data = []

# 기존 셀에서 가져온 HTML 사용
for link in soup.select("a[href]"):
    href = link.get("href", "")
    title = link.get_text(" ", strip=True)

    # 보도자료 상세 페이지 링크만 선택
    if "dtl.jsp" not in href or not title:
        continue

    # 상대경로를 전체 URL로 변환
    detail_url = urljoin(url, href)

    news_data.append({
        "제목": title,
        "상세URL": detail_url
    })

# pandas 표로 변환
news_df = pd.DataFrame(news_data)

# 같은 상세 URL이 중복되었다면 제거
news_df = news_df.drop_duplicates(
    subset=["상세URL"]
).reset_index(drop=True)

print("수집된 보도자료 수:", len(news_df))

# 최대 10건 확인
display(news_df.head(10))


수집된 보도자료 수: 10


,제목,상세URL
0,물류기업과 기술기업이 만나는 ‘협업의 장’ 열린다,https://www.molit.go.kr/USR/NEWS/m_71/dtl.jsp?...
1,"김용석 대광위원장,고양창릉 광역교통개선대책 추진상황 점검",https://www.molit.go.kr/USR/NEWS/m_71/dtl.jsp?...
2,"[차관동정] 정의경 제2차관, “수도권 서부지역 교통난 해소, 철도·도로사업 조기 ...",https://www.molit.go.kr/USR/NEWS/m_71/dtl.jsp?...
3,신도시 교통대책 속도 높인다 … 도로사업 16곳 집중 점검,https://www.molit.go.kr/USR/NEWS/m_71/dtl.jsp?...
4,"전남광주 어르신, 카드 한 장으로 도시철도 무료·교통비 환급",https://www.molit.go.kr/USR/NEWS/m_71/dtl.jsp?...
5,"김용석 대광위원장,“하남교산 주요 교통시설 적기 구축에 총력”",https://www.molit.go.kr/USR/NEWS/m_71/dtl.jsp?...
6,"문체부-국토부, 관광교통 칸막이 허물고 ‘지방관광 시대’ 연다.",https://www.molit.go.kr/USR/NEWS/m_71/dtl.jsp?...
7,3기 신도시 광역교통시설 적기 구축에 총력,https://www.molit.go.kr/USR/NEWS/m_71/dtl.jsp?...
8,제34회 물류의 날 기념식 … 물류산업의 지속가능 발전방향 모색,https://www.molit.go.kr/USR/NEWS/m_71/dtl.jsp?...
9,3기 신도시 교통대책 ‘직접 인·허가’로 속도 높인다,https://www.molit.go.kr/USR/NEWS/m_71/dtl.jsp?...


In [6]:

from urllib.parse import urlparse, parse_qs

print("===== 상세 URL 검증 =====")

# 상세 URL에서 보도자료 고유 ID 추출
news_df["보도자료ID"] = news_df["상세URL"].apply(
    lambda link: parse_qs(urlparse(link).query).get("id", [""])[0]
)

print("수집 건수:", len(news_df))
print("고유 ID 수:", news_df["보도자료ID"].nunique())
print("ID 누락:", (news_df["보도자료ID"] == "").sum())
print("중복 ID:", news_df["보도자료ID"].duplicated().sum())

display(news_df[["제목", "보도자료ID"]])


===== 상세 URL 검증 =====
수집 건수: 10
고유 ID 수: 10
ID 누락: 0
중복 ID: 0


,제목,보도자료ID
0,물류기업과 기술기업이 만나는 ‘협업의 장’ 열린다,95092592
1,"김용석 대광위원장,고양창릉 광역교통개선대책 추진상황 점검",95092593
2,"[차관동정] 정의경 제2차관, “수도권 서부지역 교통난 해소, 철도·도로사업 조기 ...",95092583
3,신도시 교통대책 속도 높인다 … 도로사업 16곳 집중 점검,95092576
4,"전남광주 어르신, 카드 한 장으로 도시철도 무료·교통비 환급",95092574
5,"김용석 대광위원장,“하남교산 주요 교통시설 적기 구축에 총력”",95092564
6,"문체부-국토부, 관광교통 칸막이 허물고 ‘지방관광 시대’ 연다.",95092547
7,3기 신도시 광역교통시설 적기 구축에 총력,95092532
8,제34회 물류의 날 기념식 … 물류산업의 지속가능 발전방향 모색,95092531
9,3기 신도시 교통대책 ‘직접 인·허가’로 속도 높인다,95092519


In [8]:

import time
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

# 시험 수집 결과 저장
detail_data = []

# 우선 보도자료 3건만 수집
sample_news = news_df.head(3)

for _, row in sample_news.iterrows():
    detail_url = row["상세URL"]

    response = requests.get(detail_url, timeout=15)
    response.raise_for_status()
    response.encoding = response.apparent_encoding

    detail_soup = BeautifulSoup(response.text, "html.parser")

    # 제목 추출
    title_tag = detail_soup.select_one("h4")
    title = title_tag.get_text(" ", strip=True) if title_tag else ""

    # 등록일과 담당부서 추출
    department = ""
    published_at = ""

    for item in detail_soup.select("ul.bd_view_ul_info li"):
        label = item.select_one("strong")
        value = item.select_one("span")

        if label is None or value is None:
            continue

        key = label.get_text(strip=True)
        text = value.get_text(" ", strip=True)

        if key == "담당부서":
            department = text
        elif key == "등록일":
            published_at = text

    # PDF 첨부파일 URL 추출
    pdf_url = ""

    for link in detail_soup.select("li.file a[href]"):
        file_name = link.get_text(" ", strip=True)

        if ".pdf" in file_name.lower():
            pdf_url = urljoin(detail_url, link.get("href"))
            break

    detail_data.append({
        "제목": title,
        "담당부서": department,
        "등록일": published_at,
        "상세URL": detail_url,
        "PDF_URL": pdf_url,
    })

    print("수집:", title)

    # 서버에 연속 요청하지 않도록 대기
    time.sleep(1)

import pandas as pd

detail_df = pd.DataFrame(detail_data)

print("\n수집 건수:", len(detail_df))
print("PDF URL 확보:", detail_df["PDF_URL"].ne("").sum())

display(detail_df)


수집: 물류기업과 기술기업이 만나는  ‘협업의 장’ 열린다
수집: 김용석 대광위원장,고양창릉 광역교통개선대책 추진상황 점검
수집: [차관동정] 정의경 제2차관, “수도권 서부지역 교통난 해소, 철도·도로사업 조기 완공에 속도 내야”

수집 건수: 3
PDF URL 확보: 3


,제목,담당부서,등록일,상세URL,PDF_URL
0,물류기업과 기술기업이 만나는 ‘협업의 장’ 열린다,물류정책과,2026-11-13 06:00,https://www.molit.go.kr/USR/NEWS/m_71/dtl.jsp?...,https://www.molit.go.kr/LCMS/DWN.jsp?fold=kore...
1,"김용석 대광위원장,고양창릉 광역교통개선대책 추진상황 점검",광역교통정책과,2026-11-12 17:41,https://www.molit.go.kr/USR/NEWS/m_71/dtl.jsp?...,https://www.molit.go.kr/LCMS/DWN.jsp?fold=kore...
2,"[차관동정] 정의경 제2차관, “수도권 서부지역 교통난 해소, 철도·도로사업 조기 ...",교통정책총괄과,2026-11-10 16:31,https://www.molit.go.kr/USR/NEWS/m_71/dtl.jsp?...,https://www.molit.go.kr/LCMS/DWN.jsp?fold=kore...


In [10]:

import requests
import time
import pandas as pd

# 기존에 수집한 보도자료 3건
pdf_check_results = []

for _, row in detail_df.iterrows():
    pdf_url = row["PDF_URL"]

    if not pdf_url:
        pdf_check_results.append({
            "제목": row["제목"],
            "HTTP상태": "URL 없음",
            "파일형식": "확인 불가",
            "PDF확인": False
        })
        continue

    try:
        # PDF 내용을 내려받되 파일로 저장하지 않음
        response = requests.get(pdf_url, timeout=30)
        response.raise_for_status()

        # PDF 파일은 일반적으로 %PDF-로 시작함
        is_pdf = response.content.startswith(b"%PDF-")

        pdf_check_results.append({
            "제목": row["제목"],
            "HTTP상태": response.status_code,
            "파일형식": response.headers.get(
                "Content-Type", "정보 없음"
            ),
            "PDF확인": is_pdf
        })

    except requests.RequestException as error:
        pdf_check_results.append({
            "제목": row["제목"],
            "HTTP상태": "오류",
            "파일형식": str(error),
            "PDF확인": False
        })

    # 서버에 과도한 요청을 보내지 않도록 대기
    time.sleep(1)

# 검사 결과를 표로 출력
pdf_check_df = pd.DataFrame(pdf_check_results)

print("검사 건수:", len(pdf_check_df))
print("정상 PDF:", pdf_check_df["PDF확인"].sum())

display(pdf_check_df)


검사 건수: 3
정상 PDF: 3


,제목,HTTP상태,파일형식,PDF확인
0,물류기업과 기술기업이 만나는 ‘협업의 장’ 열린다,200,application/x-msdownload;charset=utf-8,True
1,"김용석 대광위원장,고양창릉 광역교통개선대책 추진상황 점검",200,application/x-msdownload;charset=utf-8,True
2,"[차관동정] 정의경 제2차관, “수도권 서부지역 교통난 해소, 철도·도로사업 조기 ...",200,application/x-msdownload;charset=utf-8,True


In [11]:

import requests
import pymupdf
import time
import pandas as pd

# PDF 본문 추출 결과를 저장할 목록
extracted_data = []

for _, row in detail_df.iterrows():
    pdf_url = row["PDF_URL"]

    try:
        # PDF 파일을 메모리로 가져오기
        response = requests.get(pdf_url, timeout=30)
        response.raise_for_status()

        if not response.content.startswith(b"%PDF-"):
            print("PDF 형식 오류:", row["제목"])
            continue

        # 다운로드한 PDF를 메모리에서 열기
        with pymupdf.open(stream=response.content, filetype="pdf") as document:
            page_count = len(document)

            full_text = "\n".join(
                page.get_text("text")
                for page in document
            )

        extracted_data.append({
            "제목": row["제목"],
            "담당부서": row["담당부서"],
            "등록일": row["등록일"],
            "상세URL": row["상세URL"],
            "PDF_URL": pdf_url,
            "본문": full_text,
            "페이지수": page_count,
            "본문글자수": len(full_text),
        })

        print("추출 완료:", row["제목"])

    except (requests.RequestException, pymupdf.FileDataError) as error:
        print("추출 오류:", row["제목"], error)

    # 서버 요청 사이에 간격 유지
    time.sleep(1)

# 결과를 DataFrame으로 정리
extracted_df = pd.DataFrame(extracted_data)

print("\n===== 추출 결과 =====")
print("추출 건수:", len(extracted_df))

display(extracted_df[["제목", "페이지수", "본문글자수"]])


추출 완료: 물류기업과 기술기업이 만나는  ‘협업의 장’ 열린다
추출 완료: 김용석 대광위원장,고양창릉 광역교통개선대책 추진상황 점검
추출 완료: [차관동정] 정의경 제2차관, “수도권 서부지역 교통난 해소, 철도·도로사업 조기 완공에 속도 내야”

===== 추출 결과 =====
추출 건수: 3


,제목,페이지수,본문글자수
0,물류기업과 기술기업이 만나는 ‘협업의 장’ 열린다,6,1844
1,"김용석 대광위원장,고양창릉 광역교통개선대책 추진상황 점검",2,1035
2,"[차관동정] 정의경 제2차관, “수도권 서부지역 교통난 해소, 철도·도로사업 조기 ...",7,4065


In [17]:

# 추출된 PDF 본문 앞부분 확인
for index, row in extracted_df.iterrows():
    print(f"\n===== 보도자료 {index + 1} =====")
    print("제목:", row["제목"])
    print("페이지 수:", row["페이지수"])
    print("본문 글자 수:", row["본문글자수"])

    print("\n[본문 앞부분]")
    print(row["본문"])
    print("-" * 50)



===== 보도자료 1 =====
제목: 물류기업과 기술기업이 만나는  ‘협업의 장’ 열린다
페이지 수: 6
본문 글자 수: 1844

[본문 앞부분]
- 1 -
보도자료 
 
보도시점 : 2026. 11. 13.(금) 06:00 이후(11. 13.(금) 석간) / 배포 : 2026. 11. 12.(목)
물류기업과 기술기업이 만나는 
‘협업의 장’ 열린다
- 11월 13일 서울창업허브 공덕에서 “2026 물류상생 Meet-up” 개최
- 우수기술 발표·개방형 혁신(OI) 사례 공유·1:1 협업 매칭까지 한자리에  
□ 국토교통부(장관 홍지선)와 물류산업진흥재단(이사장 심재선)은 11월 
13일 오후 서울창업허브 공덕에서 “2026 물류상생 Meet-up”을 개최한다.
 ㅇ 물류산업의 패러다임 전환에 따라 물류 기술 도입 수요는 꾸준히 
증가하고 있으나, 물류-기술 기업 간의 협업 기회가 부족해 우수한 
신기술이 실제 현장에 적용되는 데에는 한계가 있었다. 
 ㅇ 이에 국토교통부는 정부 주도로 물류-기술기업 파트너십 구축 계기를 
마련하고, 협업을 통한 물류산업 고도화를 지원하고자 물류산업진흥
재단과 함께 “2026 물류상생 Meet-up”을 개최하게 되었다.
□ 물류상생 Meet-up에서는 기술기업이 보유하고 있는 우수 기술을 소개
하는 한편, 행사에 참여한 물류기업(수요기업)과 기술도입 및 실증을 위한 
협업 방안에 대하여 1:1로 논의할 예정이다. 
 ㅇ 아울러, 국토교통부는 우수 기술기업과 물류기업 간 협업사업이 실질
적인 사업 성과로 이어질 수 있도록 정부 지원사업과 연계해 기업 간 
협력 기반을 마련하고, 기술 도입과 현장 적용을 적극 지원할 계획이다. 
□ 이번 행사에서는 기술도입․실증을 위한 협업 파트너십 구축 성사율을 
높이기 위해 참여 물류기업(10개사)을 대상으로 기술기업과 협업을 
희망하는 8개 대과제를 사전 조사하였다. 이를 토대로 해당 기술을 
보유한 기업을 대상으로 참여 기술기업(23개사)을 선정하였다.
 
 < Meet-up 참여 

In [16]:

# 두 번째와 세 번째 보도자료만 확인
for index in [1, 2]:
    row = extracted_df.iloc[index]

    print(f"\n===== 보도자료 {index + 1} =====")
    print("제목:", row["제목"])
    print("페이지 수:", row["페이지수"])
    print("본문 글자 수:", row["본문글자수"])

    print("\n[본문 앞부분]")
    print(row["본문"])
    print("-" * 40)



===== 보도자료 2 =====
제목: 김용석 대광위원장,고양창릉 광역교통개선대책 추진상황 점검
페이지 수: 2
본문 글자 수: 1035

[본문 앞부분]
- 1 -
동정자료
보도시점 : 배포 즉시 보도 가능 / 배포 : 2026. 11. 12.(목)
김용석 대광위원장, 고양창릉 
광역교통개선대책 추진상황 점검
- 12일 오후 고양은평선 등 주요 사업 점검… 차질없는 사업추진 당부
□ 국토교통부(장관 홍지선) 대도시권광역교통위원회(이하 대광위) 김용석 
위원장은 11월 12일 오후 고양창릉 공공주택지구 광역교통개선대책 사업 
추진 상황을 점검하고, 고양시, 한국토지주택공사(LH) 등 관계기관과 주요 
현안을 논의했다.
 ㅇ 고양창릉지구는 수도권 서북부의 핵심 주거거점으로, 약 3만 8천 호의 
주택공급이 예정되어 있어 입주민의 교통편의 확보를 위한 광역교통
시설의 적기 구축이 중요한 지역이다.
□ 이날 점검회의에서는 고양창릉지구 광역교통개선대책의 주요 사업인 고양
은평선 광역철도 추진상황을 중점적으로 살펴보고, 사업 추진 과정에서의 
애로사항과 향후 일정 등을 관계기관과 함께 논의했다.
 ㅇ 고양은평선은 서울 도시철도 6호선 새절역에서 창릉신도시를 거쳐 고양
시청역까지 연결하는 광역철도 사업으로, 창릉지구 입주민의 서울 접근
성을 높이고 수도권 서북부의 광역교통 여건을 개선하는 핵심 교통이다.
□ 대광위는 최근 광역교통개선대책사업의 추진 지연과 입주 초기 교통불편에 
대한 우려가 지속적으로 제기됨에 따라 남양주왕숙지구를 시작으로 3기 
신도시 광역교통개선대책사업에 대한 추진상황을 전수점검하고 있다.
 ㅇ 이번 점검에서도 고양창릉지구의 광역교통개선대책 세부 사업별 추진
현황과 향후 일정을 확인하고 지연이 우려되는 사업에 대해서는 원인을 
분석하여 관계기관과 대응방안을 논의했다.

- 2 -
□ 김 위원장은 화정동~도내동 연결도로 확장사업 현장을 방문하여 사업 
추진상황과 현장 여건을 직접 확인하면서, 
ㅇ “사업이 계획된 일정에 맞춰 차질 없이 추진될 수 있

In [15]:
# 세 번째 보도자료만 확인
third_news = extracted_df.iloc[2]

print("제목:", third_news["제목"])
print("페이지 수:", third_news["페이지수"])
print("본문 글자 수:", third_news["본문글자수"])

print("\n===== 본문 시작 250글자 =====")
print(third_news["본문"])

제목: [차관동정] 정의경 제2차관, “수도권 서부지역 교통난 해소, 철도·도로사업 조기 완공에 속도 내야”
페이지 수: 7
본문 글자 수: 4065

===== 본문 시작 250글자 =====
- 1 -
동정자료 
보도시점 : 배포 즉시 보도 가능 / 배포 : 2026. 11. 10.(화)
정의경 제2차관, “수도권 서부지역 교통난 해소,
철도·도로사업 조기 완공에 속도 내야”
 - 10일 오후 대장~홍대 광역철도 현장에서 철도·도로사업 점검회의 및 현장 점검
□ 정의경 국토교통부 제2차관은 11월 10일 오후 대장~홍대 광역철도 
사업 현장에서 수도권 서부지역 교통 개선을 위한 철도·도로사업 추진상황 
점검회의를 주재한 후 공사 현장을 점검하였다.
ㅇ 이번 회의는 대장~홍대 광역철도, 서울 5호선 김포·검단 연장, 서부권
광역급행철도, 계양~강화 고속도로 등 수도권 서부지역 핵심 철도 및 
도로사업의 진행상황을 점검하고, 사업기간 단축 방안을 논의하기 위해 
마련되었다.
  - 특히, 최근 발표한 수도권 주택공급 대책(8.13 / 10.29)에 발맞춰 사업이 
계획대로 추진될 수 있도록 사업별 진행상황을 면밀히 점검하였다.
□ 각 사업 시행자는 사업별 진행상황과 향후 추진계획을 보고하고, 참석자들은 
사업 기간 단축 및 추진 과정의 애로사항 해소 방안을 논의하였다.
 ㅇ (대장~홍대 광역철도) 부천 대장에서 서울 홍대입구까지 총 20.1㎞ 
복선전철을 신설하는 사업으로, 올해 4월 착공 이후 토지보상, 수직구 
굴착, 지장물 이설 등 초기 공정이 활발히 진행 중이다.
  - 적기 공사 추진을 위해서는 토지 확보가 중요한 만큼, 토지보상, 점용허가 
등을 적극 지원하는 한편, 사업관리기관인 국가철도공단과 함께 철저한 
공정관리를 추진할 계획이다.

- 2 -
 ㅇ (서울 5호선 김포·검단 연장) 서울 방화역에서 인천 검단을 거쳐 김포까지 
연장하는 사업으로, 기본계획 수립(경기도, ‘26.11~) 과정에서 교통수요 
검토 및 설계 등 후속절차가 조속히 추진되

In [18]:

from pathlib import Path
import pandas as pd

# PDF에서 추출한 전체 본문 데이터
output_file = Path("trafficnews_batch_raw.csv")

# 기존 파일을 덮어쓰지 않도록 확인
if output_file.exists():
    print("이미 파일이 존재합니다. 덮어쓰지 않았습니다.")
else:
    # 전체 본문을 포함한 데이터 저장
    extracted_df.to_csv(
        output_file,
        index=False,
        encoding="utf-8-sig"
    )

    print("CSV 저장 완료!")
    print("저장 파일:", output_file)
    print("저장 건수:", len(extracted_df))

# CSV를 다시 읽어 저장 결과 확인
saved_df = pd.read_csv(output_file)

print("\n===== 저장 검증 =====")
print("CSV 데이터 수:", len(saved_df))
print("본문 누락:", saved_df["본문"].isna().sum())

# 저장 전후 본문이 완전히 같은지 확인
if len(saved_df) == len(extracted_df):
    same_text = all(
        original == saved
        for original, saved in zip(
            extracted_df["본문"],
            saved_df["본문"]
        )
    )
    print("전체 본문 일치:", same_text)
else:
    print("전체 본문 일치: 확인 불가 (데이터 수 불일치)")

display(saved_df[["제목", "페이지수", "본문글자수"]])


CSV 저장 완료!
저장 파일: trafficnews_batch_raw.csv
저장 건수: 3

===== 저장 검증 =====
CSV 데이터 수: 3
본문 누락: 0
전체 본문 일치: True


,제목,페이지수,본문글자수
0,물류기업과 기술기업이 만나는 ‘협업의 장’ 열린다,6,1844
1,"김용석 대광위원장,고양창릉 광역교통개선대책 추진상황 점검",2,1035
2,"[차관동정] 정의경 제2차관, “수도권 서부지역 교통난 해소, 철도·도로사업 조기 ...",7,4065


In [19]:

from pathlib import Path
import pandas as pd
from urllib.parse import urlparse, parse_qs

# 기존에 수집한 보도자료 1건
original_df = pd.read_csv("trafficnews_crawling.csv")

# 새로 수집한 보도자료 3건
batch_df = pd.read_csv("trafficnews_batch_raw.csv")

# URL에서 보도자료 고유 ID 추출
def get_news_id(url):
    query = parse_qs(urlparse(str(url)).query)
    return query.get("id", [""])[0]

original_ids = original_df["원본URL"].apply(get_news_id)
batch_ids = batch_df["상세URL"].apply(get_news_id)

# 두 데이터에 모두 존재하는 ID 확인
duplicate_ids = set(original_ids) & set(batch_ids)

print("기존 데이터:", len(original_df), "건")
print("추가 데이터:", len(batch_df), "건")
print("중복 건수:", len(duplicate_ids))
print("중복 ID:", sorted(duplicate_ids))

print("중복 제외 예상 전체 건수:",
      len(set(original_ids) | set(batch_ids)))


기존 데이터: 1 건
추가 데이터: 3 건
중복 건수: 0
중복 ID: []
중복 제외 예상 전체 건수: 4


In [20]:

from pathlib import Path
import pandas as pd

# 통합 원본 데이터를 저장할 파일
output_file = Path("trafficnews_all_raw.csv")

# 기존 데이터와 새 데이터를 같은 열 구조로 맞추기
original_data = original_df.rename(
    columns={"원본URL": "상세URL"}
)

batch_data = batch_df.copy()

# 두 데이터 합치기
combined_df = pd.concat(
    [original_data, batch_data],
    ignore_index=True
)

# 보도자료 고유 ID 추출
combined_df["보도자료ID"] = combined_df["상세URL"].apply(get_news_id)

# 중복 ID 제거
combined_df = combined_df.drop_duplicates(
    subset=["보도자료ID"]
).reset_index(drop=True)

# 기존 파일 덮어쓰기 방지
if output_file.exists():
    print("이미 통합 CSV 파일이 존재합니다. 덮어쓰지 않았습니다.")
else:
    combined_df.to_csv(
        output_file,
        index=False,
        encoding="utf-8-sig"
    )
    print("통합 CSV 저장 완료!")

# 저장된 파일 검증
saved_df = pd.read_csv(output_file)

print("통합 데이터 수:", len(saved_df))
print("본문 누락:", saved_df["본문"].isna().sum())
print("중복 ID:", saved_df["보도자료ID"].duplicated().sum())

display(saved_df[["보도자료ID", "제목", "등록일"]])


통합 CSV 저장 완료!
통합 데이터 수: 4
본문 누락: 0
중복 ID: 0


,보도자료ID,제목,등록일
0,95092576,신도시 교통대책 속도 높인다 … 도로사업 16곳 집중 점검,2026-11-10 11:00
1,95092592,물류기업과 기술기업이 만나는 ‘협업의 장’ 열린다,2026-11-13 06:00
2,95092593,"김용석 대광위원장,고양창릉 광역교통개선대책 추진상황 점검",2026-11-12 17:41
3,95092583,"[차관동정] 정의경 제2차관, “수도권 서부지역 교통난 해소, 철도·도로사업 조기 ...",2026-11-10 16:31


In [21]:

import pandas as pd

# 통합 원본 CSV 읽기
df = pd.read_csv("trafficnews_all_raw.csv")

print("===== 통합 데이터 구조 검증 =====")
print("전체 데이터:", len(df), "건")
print("전체 열:", len(df.columns), "개")

# 열별 누락 개수 계산
missing_count = df.isna().sum()

# 빈 문자열도 누락으로 계산
empty_count = df.fillna("").astype(str).apply(
    lambda column: column.str.strip().eq("").sum()
)

result = pd.DataFrame({
    "열 이름": df.columns,
    "누락 건수": missing_count.values,
    "빈 값 포함 누락 건수": empty_count.values
})

display(result)


===== 통합 데이터 구조 검증 =====
전체 데이터: 4 건
전체 열: 9 개


C:\Users\EZ\AppData\Local\Temp\ipykernel_14632\2588264925.py:14: Pandas4Warning: 'str' is not supported as a fill value for float64 dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  empty_count = df.fillna("").astype(str).apply(


,열 이름,누락 건수,빈 값 포함 누락 건수
0,제목,0,0
1,담당부서,0,0
2,등록일,0,0
3,상세URL,0,0
4,PDF_URL,0,0
5,본문,0,0
6,페이지수,1,1
7,본문글자수,1,1
8,보도자료ID,0,0


In [22]:

import pandas as pd

# 통합 원본 데이터 불러오기
df = pd.read_csv("trafficnews_all_raw.csv")

# 기존에 검증한 첫 번째 보도자료 ID
target_id = 95092576

# 해당 보도자료 찾기
target = df["보도자료ID"].astype(str) == str(target_id)

if target.sum() == 1:
    # PDF 원본에서 확인한 페이지 수
    df.loc[target, "페이지수"] = 2

    # 저장된 본문을 기준으로 글자 수 재계산
    df.loc[target, "본문글자수"] = df.loc[
        target, "본문"
    ].str.len()

    print("기존 보도자료 정보 보완 완료")
else:
    print("대상 보도자료를 정확히 1건 찾지 못했습니다.")

print("\n===== 누락 확인 =====")
print(df[["페이지수", "본문글자수"]].isna().sum())

display(df[["보도자료ID", "제목", "페이지수", "본문글자수"]])


기존 보도자료 정보 보완 완료

===== 누락 확인 =====
페이지수     0
본문글자수    0
dtype: int64


,보도자료ID,제목,페이지수,본문글자수
0,95092576,신도시 교통대책 속도 높인다 … 도로사업 16곳 집중 점검,2.0,1707.0
1,95092592,물류기업과 기술기업이 만나는 ‘협업의 장’ 열린다,6.0,1844.0
2,95092593,"김용석 대광위원장,고양창릉 광역교통개선대책 추진상황 점검",2.0,1035.0
3,95092583,"[차관동정] 정의경 제2차관, “수도권 서부지역 교통난 해소, 철도·도로사업 조기 ...",7.0,4065.0


In [23]:

from pathlib import Path
import pandas as pd

# 검증된 통합 데이터 저장 경로
output_file = Path("trafficnews_all_verified.csv")

# 페이지 수와 본문 글자 수를 정수형으로 정리
verified_df = df.copy()

verified_df["페이지수"] = verified_df["페이지수"].astype(int)
verified_df["본문글자수"] = verified_df["본문글자수"].astype(int)

# 기존 파일 덮어쓰기 방지
if output_file.exists():
    print("이미 파일이 존재합니다. 덮어쓰지 않았습니다.")
else:
    verified_df.to_csv(
        output_file,
        index=False,
        encoding="utf-8-sig"
    )
    print("검증 데이터 저장 완료!")

# 저장 결과 확인
saved_verified = pd.read_csv(output_file)

print("저장 건수:", len(saved_verified))
print("본문 누락:", saved_verified["본문"].isna().sum())
print("중복 ID:", saved_verified["보도자료ID"].duplicated().sum())

# 본문 전체 일치 여부 확인
print(
    "본문 전체 일치:",
    saved_verified["본문"].tolist() == verified_df["본문"].tolist()
)

display(
    saved_verified[
        ["보도자료ID", "제목", "페이지수", "본문글자수"]
    ]
)


검증 데이터 저장 완료!
저장 건수: 4
본문 누락: 0
중복 ID: 0
본문 전체 일치: True


,보도자료ID,제목,페이지수,본문글자수
0,95092576,신도시 교통대책 속도 높인다 … 도로사업 16곳 집중 점검,2,1707
1,95092592,물류기업과 기술기업이 만나는 ‘협업의 장’ 열린다,6,1844
2,95092593,"김용석 대광위원장,고양창릉 광역교통개선대책 추진상황 점검",2,1035
3,95092583,"[차관동정] 정의경 제2차관, “수도권 서부지역 교통난 해소, 철도·도로사업 조기 ...",7,4065


In [24]:

# 기존에 수집한 보도자료 ID
collected_ids = set(
    saved_verified["보도자료ID"].astype(str)
)

# 목록에 있는 보도자료 중 아직 수집하지 않은 자료만 선택
remaining_df = news_df[
    ~news_df["보도자료ID"].astype(str).isin(collected_ids)
].copy()

print("목록의 전체 보도자료:", len(news_df))
print("이미 수집한 보도자료:", len(collected_ids))
print("새로운 수집 대상:", len(remaining_df))

display(
    remaining_df[["보도자료ID", "제목", "상세URL"]]
)


목록의 전체 보도자료: 10
이미 수집한 보도자료: 4
새로운 수집 대상: 6


,보도자료ID,제목,상세URL
4,95092574,"전남광주 어르신, 카드 한 장으로 도시철도 무료·교통비 환급",https://www.molit.go.kr/USR/NEWS/m_71/dtl.jsp?...
5,95092564,"김용석 대광위원장,“하남교산 주요 교통시설 적기 구축에 총력”",https://www.molit.go.kr/USR/NEWS/m_71/dtl.jsp?...
6,95092547,"문체부-국토부, 관광교통 칸막이 허물고 ‘지방관광 시대’ 연다.",https://www.molit.go.kr/USR/NEWS/m_71/dtl.jsp?...
7,95092532,3기 신도시 광역교통시설 적기 구축에 총력,https://www.molit.go.kr/USR/NEWS/m_71/dtl.jsp?...
8,95092531,제34회 물류의 날 기념식 … 물류산업의 지속가능 발전방향 모색,https://www.molit.go.kr/USR/NEWS/m_71/dtl.jsp?...
9,95092519,3기 신도시 교통대책 ‘직접 인·허가’로 속도 높인다,https://www.molit.go.kr/USR/NEWS/m_71/dtl.jsp?...


In [25]:

import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

new_detail_data = []

# 아직 수집하지 않은 6건만 처리
for _, row in remaining_df.iterrows():
    detail_url = row["상세URL"]

    try:
        response = requests.get(detail_url, timeout=20)
        response.raise_for_status()
        response.encoding = response.apparent_encoding

        detail_soup = BeautifulSoup(
            response.text, "html.parser"
        )

        # 제목 추출
        title_tag = detail_soup.select_one("h4")
        title = (
            title_tag.get_text(" ", strip=True)
            if title_tag else ""
        )

        # 담당부서와 등록일 추출
        department = ""
        published_at = ""

        for item in detail_soup.select("ul.bd_view_ul_info li"):
            label = item.select_one("strong")
            value = item.select_one("span")

            if label is None or value is None:
                continue

            key = label.get_text(strip=True)
            text = value.get_text(" ", strip=True)

            if key == "담당부서":
                department = text
            elif key == "등록일":
                published_at = text

        # PDF 다운로드 URL 추출
        pdf_url = ""

        for link in detail_soup.select("li.file a[href]"):
            file_name = link.get_text(" ", strip=True)

            if ".pdf" in file_name.lower():
                pdf_url = urljoin(detail_url, link.get("href"))
                break

        new_detail_data.append({
            "보도자료ID": row["보도자료ID"],
            "제목": title,
            "담당부서": department,
            "등록일": published_at,
            "상세URL": detail_url,
            "PDF_URL": pdf_url,
        })

        print("수집 완료:", title)

    except requests.RequestException as error:
        print("수집 오류:", row["보도자료ID"], error)

    # 서버 요청 간격 유지
    time.sleep(1)

# 결과 확인
new_detail_df = pd.DataFrame(new_detail_data)

print("\n===== 수집 결과 =====")
print("수집 건수:", len(new_detail_df))
print("PDF URL 확보:", new_detail_df["PDF_URL"].ne("").sum())

display(new_detail_df)


수집 완료: 전남광주 어르신, 카드 한 장으로 도시철도 무료·교통비 환급
수집 완료: 김용석 대광위원장,“하남교산 주요 교통시설 적기 구축에 총력”
수집 완료: 문체부-국토부, 관광교통 칸막이 허물고 ‘지방관광 시대’ 연다.
수집 완료: 3기 신도시 광역교통시설 적기 구축에 총력
수집 완료: 제34회 물류의 날 기념식 … 물류산업의 지속가능 발전방향 모색
수집 완료: 3기 신도시 교통대책  ‘직접 인·허가’로 속도 높인다

===== 수집 결과 =====
수집 건수: 6
PDF URL 확보: 6


,보도자료ID,제목,담당부서,등록일,상세URL,PDF_URL
0,95092574,"전남광주 어르신, 카드 한 장으로 도시철도 무료·교통비 환급",광역교통경제과,2026-11-09 15:30,https://www.molit.go.kr/USR/NEWS/m_71/dtl.jsp?...,https://www.molit.go.kr/LCMS/DWN.jsp?fold=kore...
1,95092564,"김용석 대광위원장,“하남교산 주요 교통시설 적기 구축에 총력”",광역교통정책과,2026-11-05 16:40,https://www.molit.go.kr/USR/NEWS/m_71/dtl.jsp?...,https://www.molit.go.kr/LCMS/DWN.jsp?fold=kore...
2,95092547,"문체부-국토부, 관광교통 칸막이 허물고 ‘지방관광 시대’ 연다.",교통정책총괄과,2026-11-02 16:00,https://www.molit.go.kr/USR/NEWS/m_71/dtl.jsp?...,https://www.molit.go.kr/LCMS/DWN.jsp?fold=kore...
3,95092532,3기 신도시 광역교통시설 적기 구축에 총력,광역교통정책과,2026-10-28 14:00,https://www.molit.go.kr/USR/NEWS/m_71/dtl.jsp?...,https://www.molit.go.kr/LCMS/DWN.jsp?fold=kore...
4,95092531,제34회 물류의 날 기념식 … 물류산업의 지속가능 발전방향 모색,물류정책과,2026-10-28 11:00,https://www.molit.go.kr/USR/NEWS/m_71/dtl.jsp?...,https://www.molit.go.kr/LCMS/DWN.jsp?fold=kore...
5,95092519,3기 신도시 교통대책 ‘직접 인·허가’로 속도 높인다,광역교통정책과,2026-10-25 11:00,https://www.molit.go.kr/USR/NEWS/m_71/dtl.jsp?...,https://www.molit.go.kr/LCMS/DWN.jsp?fold=kore...


In [27]:

import time
import requests
import pymupdf
import pandas as pd

# 신규 보도자료 PDF 본문 추출
new_extracted_data = []

for _, row in new_detail_df.iterrows():
    try:
        response = requests.get(row["PDF_URL"], timeout=30)
        response.raise_for_status()

        # PDF 파일 형식 확인
        if not response.content.startswith(b"%PDF-"):
            print("PDF 형식 오류:", row["제목"])
            continue

        # PDF 전체 페이지에서 본문 추출
        with pymupdf.open(
            stream=response.content,
            filetype="pdf"
        ) as document:
            page_count = len(document)

            full_text = "\n".join(
                page.get_text("text")
                for page in document
            )

        new_extracted_data.append({
            "보도자료ID": row["보도자료ID"],
            "제목": row["제목"],
            "담당부서": row["담당부서"],
            "등록일": row["등록일"],
            "상세URL": row["상세URL"],
            "PDF_URL": row["PDF_URL"],
            "본문": full_text,
            "페이지수": page_count,
            "본문글자수": len(full_text),
        })

        print("추출 완료:", row["제목"])

    except (requests.RequestException, pymupdf.FileDataError) as error:
        print("추출 오류:", row["제목"], error)

    # 서버 요청 간격 유지
    time.sleep(1)

# 추출 결과 확인
new_extracted_df = pd.DataFrame(new_extracted_data)

print("\n===== 신규 PDF 추출 결과 =====")
print("추출 건수:", len(new_extracted_df))
print("본문 누락:", new_extracted_df["본문"].isna().sum())

display(
    new_extracted_df[
        ["보도자료ID", "제목", "페이지수", "본문글자수"]
    ]
)


추출 완료: 전남광주 어르신, 카드 한 장으로 도시철도 무료·교통비 환급
추출 완료: 김용석 대광위원장,“하남교산 주요 교통시설 적기 구축에 총력”
추출 완료: 문체부-국토부, 관광교통 칸막이 허물고 ‘지방관광 시대’ 연다.
추출 완료: 3기 신도시 광역교통시설 적기 구축에 총력
추출 완료: 제34회 물류의 날 기념식 … 물류산업의 지속가능 발전방향 모색
추출 완료: 3기 신도시 교통대책  ‘직접 인·허가’로 속도 높인다

===== 신규 PDF 추출 결과 =====
추출 건수: 6
본문 누락: 0


,보도자료ID,제목,페이지수,본문글자수
0,95092574,"전남광주 어르신, 카드 한 장으로 도시철도 무료·교통비 환급",3,2753
1,95092564,"김용석 대광위원장,“하남교산 주요 교통시설 적기 구축에 총력”",2,1391
2,95092547,"문체부-국토부, 관광교통 칸막이 허물고 ‘지방관광 시대’ 연다.",2,1460
3,95092532,3기 신도시 광역교통시설 적기 구축에 총력,2,1631
4,95092531,제34회 물류의 날 기념식 … 물류산업의 지속가능 발전방향 모색,8,5694
5,95092519,3기 신도시 교통대책 ‘직접 인·허가’로 속도 높인다,2,1512


In [28]:

import pandas as pd

# 기존 검증 데이터 4건
existing_df = pd.read_csv("trafficnews_all_verified.csv")

# 신규 추출 데이터 6건
additional_df = new_extracted_df.copy()

# 기존 4건과 신규 6건 결합
all_news_df = pd.concat(
    [existing_df, additional_df],
    ignore_index=True
)

# 보도자료 ID 기준 중복 검사
duplicate_count = all_news_df["보도자료ID"].duplicated().sum()

# 본문 누락 검사
missing_count = all_news_df["본문"].isna().sum()

print("===== 통합 결과 =====")
print("전체 보도자료:", len(all_news_df))
print("중복 ID:", duplicate_count)
print("본문 누락:", missing_count)
print("전체 본문 글자 수:", all_news_df["본문"].str.len().sum())

display(
    all_news_df[
        ["보도자료ID", "제목", "페이지수", "본문글자수"]
    ]
)


===== 통합 결과 =====
전체 보도자료: 10
중복 ID: 0
본문 누락: 0
전체 본문 글자 수: 23092


,보도자료ID,제목,페이지수,본문글자수
0,95092576,신도시 교통대책 속도 높인다 … 도로사업 16곳 집중 점검,2,1707
1,95092592,물류기업과 기술기업이 만나는 ‘협업의 장’ 열린다,6,1844
2,95092593,"김용석 대광위원장,고양창릉 광역교통개선대책 추진상황 점검",2,1035
3,95092583,"[차관동정] 정의경 제2차관, “수도권 서부지역 교통난 해소, 철도·도로사업 조기 ...",7,4065
4,95092574,"전남광주 어르신, 카드 한 장으로 도시철도 무료·교통비 환급",3,2753
5,95092564,"김용석 대광위원장,“하남교산 주요 교통시설 적기 구축에 총력”",2,1391
6,95092547,"문체부-국토부, 관광교통 칸막이 허물고 ‘지방관광 시대’ 연다.",2,1460
7,95092532,3기 신도시 광역교통시설 적기 구축에 총력,2,1631
8,95092531,제34회 물류의 날 기념식 … 물류산업의 지속가능 발전방향 모색,8,5694
9,95092519,3기 신도시 교통대책 ‘직접 인·허가’로 속도 높인다,2,1512


In [29]:

from pathlib import Path
import pandas as pd

# 통합 원본 데이터 저장 경로
output_file = Path("trafficnews_all_10_raw.csv")

# 중복과 본문 누락 재확인
duplicate_count = all_news_df["보도자료ID"].duplicated().sum()
missing_count = all_news_df["본문"].isna().sum()

if duplicate_count > 0 or missing_count > 0:
    print("중복 또는 본문 누락이 있어 저장하지 않습니다.")

elif output_file.exists():
    print("이미 파일이 존재하여 덮어쓰지 않았습니다.")

else:
    # 전체 본문을 그대로 저장
    all_news_df.to_csv(
        output_file,
        index=False,
        encoding="utf-8-sig"
    )
    print("통합 원본 CSV 저장 완료!")

# 저장된 파일이 있을 때만 검증
if output_file.exists():
    saved_df = pd.read_csv(output_file)

    print("저장 건수:", len(saved_df))
    print("중복 ID:", saved_df["보도자료ID"].duplicated().sum())
    print("본문 누락:", saved_df["본문"].isna().sum())

    # 저장 전후 모든 본문 비교
    same_text = saved_df["본문"].tolist() == all_news_df["본문"].tolist()
    print("본문 전체 일치:", same_text)

    print("저장 파일:", output_file)


통합 원본 CSV 저장 완료!
저장 건수: 10
중복 ID: 0
본문 누락: 0
본문 전체 일치: True
저장 파일: trafficnews_all_10_raw.csv


In [30]:

import re
import pandas as pd

# 저장된 통합 원본 CSV 불러오기
quality_df = pd.read_csv("trafficnews_all_10_raw.csv")

# 점검 결과를 저장할 목록
quality_results = []

for _, row in quality_df.iterrows():
    text = str(row["본문"])

    quality_results.append({
        "보도자료ID": row["보도자료ID"],
        "제목": row["제목"],
        "본문글자수": len(text),
        "짧은본문": len(text.strip()) < 300,
        "전화번호포함": bool(
            re.search(r"0\d{1,2}-\d{3,4}-\d{4}", text)
        ),
        "페이지번호포함": bool(
            re.search(r"(?m)^\s*-\s*\d+\s*-\s*$", text)
        ),
        "특수문자포함": "\uf000" in text,
    })

# 점검 결과를 표로 변환
quality_result_df = pd.DataFrame(quality_results)

print("===== 데이터 품질 검사 =====")
print("전체 자료:", len(quality_result_df))
print("짧은 본문:", quality_result_df["짧은본문"].sum())
print("전화번호 포함:", quality_result_df["전화번호포함"].sum())
print("페이지 번호 포함:", quality_result_df["페이지번호포함"].sum())
print("특수문자 포함:", quality_result_df["특수문자포함"].sum())

display(quality_result_df)


===== 데이터 품질 검사 =====
전체 자료: 10
짧은 본문: 0
전화번호 포함: 10
페이지 번호 포함: 10
특수문자 포함: 1


,보도자료ID,제목,본문글자수,짧은본문,전화번호포함,페이지번호포함,특수문자포함
0,95092576,신도시 교통대책 속도 높인다 … 도로사업 16곳 집중 점검,1707,False,True,True,True
1,95092592,물류기업과 기술기업이 만나는 ‘협업의 장’ 열린다,1844,False,True,True,False
2,95092593,"김용석 대광위원장,고양창릉 광역교통개선대책 추진상황 점검",1035,False,True,True,False
3,95092583,"[차관동정] 정의경 제2차관, “수도권 서부지역 교통난 해소, 철도·도로사업 조기 ...",4065,False,True,True,False
4,95092574,"전남광주 어르신, 카드 한 장으로 도시철도 무료·교통비 환급",2753,False,True,True,False
5,95092564,"김용석 대광위원장,“하남교산 주요 교통시설 적기 구축에 총력”",1391,False,True,True,False
6,95092547,"문체부-국토부, 관광교통 칸막이 허물고 ‘지방관광 시대’ 연다.",1460,False,True,True,False
7,95092532,3기 신도시 광역교통시설 적기 구축에 총력,1631,False,True,True,False
8,95092531,제34회 물류의 날 기념식 … 물류산업의 지속가능 발전방향 모색,5694,False,True,True,False
9,95092519,3기 신도시 교통대책 ‘직접 인·허가’로 속도 높인다,1512,False,True,True,False


In [31]:

# 특수문자가 발견된 보도자료 확인
target_character = "\uf000"

for _, row in quality_df.iterrows():
    text = row["본문"]

    if target_character not in text:
        continue

    print("보도자료 ID:", row["보도자료ID"])
    print("제목:", row["제목"])
    print("발견 횟수:", text.count(target_character))

    print("\n===== 특수문자가 포함된 줄 =====")

    for line_number, line in enumerate(text.splitlines(), start=1):
        if target_character in line:
            print(f"{line_number}행: {line}")


보도자료 ID: 95092576
제목: 신도시 교통대책 속도 높인다 … 도로사업 16곳 집중 점검
발견 횟수: 6

===== 특수문자가 포함된 줄 =====
21행:  (하남교산) 객산터널~국도43호선 연결도로,  (인천계양) 국도 39호선 연계도로
22행:  (과천과천) 과천~우면산간 도시고속화도로 지하화, 상아벌 지하차도 확장 및 통합
23행:  (안산장상) 지하차도 3개소(수암동 우체국 앞, 신설 교차로, 제일CC 사거리)
24행:  (의왕군포안산) 안산시 해안로 확장, 사업지~수인로 연결도로, 용담IC 연결로 신설
25행:  (시흥장현‧시흥목감) 죽율~장현~목감간 도로 개설, 장현 교차로, 산현 교차로. 


In [32]:

import pandas as pd

# 원본 데이터를 복사하여 정제본 준비
cleaned_df = quality_df.copy()

# PDF에서 깨진 목록 기호를 읽기 쉬운 기호로 교체
cleaned_df["본문"] = cleaned_df["본문"].str.replace(
    "\uf000", "◆", regex=False
)

# 특수문자 교체 결과 확인
remaining_count = cleaned_df["본문"].str.count(
    "\uf000"
).sum()

replacement_count = cleaned_df["본문"].str.count(
    "◆"
).sum()

print("남은 특수문자:", remaining_count)
print("교체된 목록 기호:", replacement_count)

# 해당 보도자료의 목록 확인
target_text = cleaned_df.loc[
    cleaned_df["보도자료ID"] == 95092576, "본문"
].iloc[0]

for line in target_text.splitlines():
    if "◆" in line:
        print(line)


남은 특수문자: 0
교체된 목록 기호: 6
◆ (하남교산) 객산터널~국도43호선 연결도로, ◆ (인천계양) 국도 39호선 연계도로
◆ (과천과천) 과천~우면산간 도시고속화도로 지하화, 상아벌 지하차도 확장 및 통합
◆ (안산장상) 지하차도 3개소(수암동 우체국 앞, 신설 교차로, 제일CC 사거리)
◆ (의왕군포안산) 안산시 해안로 확장, 사업지~수인로 연결도로, 용담IC 연결로 신설
◆ (시흥장현‧시흥목감) 죽율~장현~목감간 도로 개설, 장현 교차로, 산현 교차로. 


In [33]:

import re

# PDF 페이지 번호 형식: - 1 -, - 2 - 등
page_pattern = r"(?m)^[ \t]*-[ \t]*\d+[ \t]*-[ \t]*$"

# 제거 전 페이지 번호 개수
before_count = cleaned_df["본문"].apply(
    lambda text: len(re.findall(page_pattern, text))
).sum()

# 페이지 번호가 있는 줄만 제거
cleaned_df["본문"] = cleaned_df["본문"].apply(
    lambda text: re.sub(page_pattern, "", text)
)

# 제거 후 페이지 번호 개수
after_count = cleaned_df["본문"].apply(
    lambda text: len(re.findall(page_pattern, text))
).sum()

print("제거 전 페이지 번호:", before_count)
print("제거 후 페이지 번호:", after_count)
print("데이터 수:", len(cleaned_df))
print("본문 누락:", cleaned_df["본문"].isna().sum())


제거 전 페이지 번호: 36
제거 후 페이지 번호: 0
데이터 수: 10
본문 누락: 0


In [35]:

import re

# 일반적인 국내 전화번호 형식
phone_pattern = r"0\d{1,2}-\d{3,4}-\d{4}"

for _, row in cleaned_df.iterrows():
    text = row["본문"]

    phone_lines = [
        (line_number, line)
        for line_number, line in enumerate(text.splitlines(), start=1)
        if re.search(phone_pattern, line)
    ]

    print("\n보도자료 ID:", row["보도자료ID"])
    print("제목:", row["제목"])
    print("전화번호 포함 줄:", len(phone_lines))

    for line_number, line in phone_lines:
        print(f"{line_number}행: {line}")


보도자료 ID: 95092576
제목: 신도시 교통대책 속도 높인다 … 도로사업 16곳 집중 점검
전화번호 포함 줄: 3
51행: (044-201-5045)
55행: (044-201-5046)
58행: (044-201-5048)

보도자료 ID: 95092592
제목: 물류기업과 기술기업이 만나는  ‘협업의 장’ 열린다
전화번호 포함 줄: 2
58행: 손덕환(044-201-3993)
61행: 박승연(044-201-3995)

보도자료 ID: 95092593
제목: 김용석 대광위원장,고양창릉 광역교통개선대책 추진상황 점검
전화번호 포함 줄: 3
36행: 과  장문석준(044-201-5045)
38행: 사무관임종채(044-201-5050)
39행: 주무관백세영(044-201-5051)

보도자료 ID: 95092583
제목: [차관동정] 정의경 제2차관, “수도권 서부지역 교통난 해소, 철도·도로사업 조기 완공에 속도 내야”
전화번호 포함 줄: 13
58행: 044-201-3804
62행: 044-201-3795
68행: 044-201-3988
72행: 044-201-4633
75행: 044-201-3984
81행: 044-201-5100
85행: 044-201-5104
91행: 044-201-4166
95행: 044-201-4162
101행: 044-201-3888
105행: 044-201-3889
111행: 044-201-4441
115행: 044-201-4556

보도자료 ID: 95092574
제목: 전남광주 어르신, 카드 한 장으로 도시철도 무료·교통비 환급
전화번호 포함 줄: 11
75행: 천재민(044-201-5080)
78행: 이미래(044-201-5087)
80행: 박영서(044-201-5022)
85행: 배상영(062-613-4500)
88행: 주보성(062-613-5830)
90행: 기창용(062-613-5831)
95행: 조택관(054-459-7260)
98행: 김민석(054-459-7441)
100행: 강민승(054-459-7446)
10

In [36]:

import re

# 확인할 보도자료
target_id = 95092583

target_text = cleaned_df.loc[
    cleaned_df["보도자료ID"] == target_id,
    "본문"
].iloc[0]

lines = target_text.splitlines()
phone_pattern = r"0\d{1,2}[- ]\d{3,4}[- ]\d{4}"

# 전화번호가 있는 줄과 주변 문맥 확인
for index, line in enumerate(lines):
    if re.search(phone_pattern, line):
        print(f"\n===== 전화번호 발견: {index + 1}행 =====")

        start = max(0, index - 2)
        end = min(len(lines), index + 3)

        for position in range(start, end):
            print(f"{position + 1}행: {lines[position]}")



===== 전화번호 발견: 58행 =====
56행: 과 장
57행: 안광열
58행: 044-201-3804
59행: 담당자
60행: 주무관

===== 전화번호 발견: 62행 =====
60행: 주무관
61행: 안규빈
62행: 044-201-3795
63행: 철도투자개발과
64행: (대장~홍대 광역철도)

===== 전화번호 발견: 68행 =====
66행: 과 장
67행: 이중기
68행: 044-201-3988
69행: 담당자
70행: 사무관

===== 전화번호 발견: 72행 =====
70행: 사무관
71행: 신용화
72행: 044-201-4633
73행: 주무관
74행: 이희운

===== 전화번호 발견: 75행 =====
73행: 주무관
74행: 이희운
75행: 044-201-3984
76행: 광역시설정책과
77행: (서울5호선 김포‧검단 연장)

===== 전화번호 발견: 81행 =====
79행: 과 장
80행: 박문수
81행: 044-201-5100
82행: 담당자
83행: 사무관

===== 전화번호 발견: 85행 =====
83행: 사무관
84행: 이경록
85행: 044-201-5104
86행: 광역급행철도기획과
87행: (서부권 광역급행철도)

===== 전화번호 발견: 91행 =====
89행: 과 장
90행: 박우성
91행: 044-201-4166
92행: 담당자
93행: 사무관

===== 전화번호 발견: 95행 =====
93행: 사무관
94행: 박동감
95행: 044-201-4162
96행: 도로건설과
97행: (계양~강화 고속도로)

===== 전화번호 발견: 101행 =====
99행: 과장
100행: 한성수
101행: 044-201-3888
102행: 담당자
103행: 사무관

===== 전화번호 발견: 105행 =====
103행: 사무관
104행: 오요셉
105행: 044-201-3889
106행: 공공택지기획과
107행: (김포아라 공공주택지구)

===== 전화번호 발견: 111행 =====
109행: 과 장
110

In [37]:

import re
import pandas as pd

# 담당자 영역 후보 확인
footer_results = []

for _, row in cleaned_df.iterrows():
    lines = row["본문"].splitlines()

    # 문서 끝부분에서 '담당부서'가 시작되는 위치 탐색
    footer_positions = [
        index
        for index, line in enumerate(lines)
        if line.strip() == "담당부서"
    ]

    footer_results.append({
        "보도자료ID": row["보도자료ID"],
        "전체줄수": len(lines),
        "담당부서위치": (
            footer_positions[-1] + 1
            if footer_positions else None
        ),
        "담당부서이후줄수": (
            len(lines) - footer_positions[-1]
            if footer_positions else None
        ),
    })

footer_df = pd.DataFrame(footer_results)

display(footer_df)


,보도자료ID,전체줄수,담당부서위치,담당부서이후줄수
0,95092576,58,45.0,14.0
1,95092592,73,NaN,NaN
2,95092593,39,NaN,NaN
3,95092583,187,NaN,NaN
4,95092574,108,NaN,NaN
5,95092564,49,NaN,NaN
6,95092547,59,NaN,NaN
7,95092532,56,NaN,NaN
8,95092531,506,NaN,NaN
9,95092519,56,43.0,14.0


In [38]:

import re
import pandas as pd

phone_pattern = r"0\d{1,2}[- ]\d{3,4}[- ]\d{4}"

contact_results = []

for _, row in cleaned_df.iterrows():
    lines = row["본문"].splitlines()

    phone_positions = [
        index
        for index, line in enumerate(lines)
        if re.search(phone_pattern, line)
    ]

    contact_results.append({
        "보도자료ID": row["보도자료ID"],
        "전체줄수": len(lines),
        "첫전화번호위치": (
            phone_positions[0] + 1 if phone_positions else None
        ),
        "마지막전화번호위치": (
            phone_positions[-1] + 1 if phone_positions else None
        ),
        "전화번호포함줄수": len(phone_positions),
        "마지막전화번호이후줄수": (
            len(lines) - phone_positions[-1] - 1
            if phone_positions else None
        ),
    })

contact_df = pd.DataFrame(contact_results)

display(contact_df)


,보도자료ID,전체줄수,첫전화번호위치,마지막전화번호위치,전화번호포함줄수,마지막전화번호이후줄수
0,95092576,58,51,58,3,0
1,95092592,73,58,61,2,12
2,95092593,39,36,39,3,0
3,95092583,187,58,115,13,72
4,95092574,108,75,108,11,0
5,95092564,49,46,49,3,0
6,95092547,59,46,59,4,0
7,95092532,56,52,56,3,0
8,95092531,506,61,74,5,432
9,95092519,56,49,56,3,0


In [39]:

import re

target_id = 95092531

target_text = cleaned_df.loc[
    cleaned_df["보도자료ID"] == target_id,
    "본문"
].iloc[0]

lines = target_text.splitlines()

phone_pattern = r"0\d{1,2}[- ]\d{3,4}[- ]\d{4}"

phone_positions = [
    index
    for index, line in enumerate(lines)
    if re.search(phone_pattern, line)
]

if phone_positions:
    last_phone_index = phone_positions[-1]

    print("전체 줄 수:", len(lines))
    print("마지막 전화번호 위치:", last_phone_index + 1)

    print("\n===== 전화번호 이후 전체 내용 =====")

    for index in range(last_phone_index + 1, len(lines)):
        print(f"{index + 1}행: {lines[index]}")
else:
    print("전화번호를 찾지 못했습니다.")


전체 줄 수: 506
마지막 전화번호 위치: 74

===== 전화번호 이후 전체 내용 =====
75행: 
76행: 
77행: 참고 1  정부포상 대상 명단(13명)
78행: 훈격
79행: 소속
80행: (직위)
81행: 성명
82행: (생년)
83행: 공적개요
84행: 훈장
85행: ㈜한국비즈넷
86행: (대표이사)
87행: 김용필
88행: (‘49년生)
89행: 국내최초로국제화물운송주선업전용정보시스템을
90행: 개발하여1,500여개중소물류기업의디지털전환을
91행: 지원하는등물류산업생산성향상에기여
92행: 산업
93행: 포장
94행: 동원물류㈜
95행: (대표이사)
96행: 신경용
97행: (‘70년生)
98행: 콜드체인기반3PL 물류체계구축을위해기흥‧안
99행: 성‧용인에농수산물종합유통센터등수도권신선물
100행: 류인프라확충에참여하는등국내신선식품물류
101행: 체계구축에기여
102행: 산업
103행: 포장
104행: 한국교통
105행: 연구원
106행: (선임연구위원)
107행: 서상범
108행: (‘68년生)
109행: 국내주요물류정책수립및제도개선연구에책임
110행: 자로참여하여110여건의정책연구를통해물류산
111행: 업발전에기여
112행: 대통령
113행: 표창
114행: ㈜광진
115행: 티엘에스
116행: (대표이사)
117행: 김경호
118행: (‘61년生)
119행: 부산‧창원‧진해‧의왕‧구미‧강릉등국내주요물류거
120행: 점을구축하고, 다년간제조기업대상물류컨설팅
121행: 과대형중량물운송프로젝트를수행하여제
122행: 조업의물류비용절감에기여
123행: 대통령
124행: 표창
125행: 와이에스엘
126행: 에이전시(주)
127행: (대표이사)
128행: 조동제
129행: (‘61년生)
130행: 기존광양항에서출발하는군산시소재기업의부
131행: 분조립상태자동차및완전분해부품의수출경
132행: 로를군산항으로전환하는운송경로를개발하여상
133행: 해행수출화물의내륙물류비절감에기여하는등
134행: 물류산업발전에기여
135행: 대통령
136행: 표창
137행

In [40]:

# 보도자료 95092531의 참고자료 시작 위치 확인

target_text = cleaned_df.loc[
    cleaned_df["보도자료ID"] == 95092531,
    "본문"
].iloc[0]

lines = target_text.splitlines()

# '참고 1'이 포함된 줄 찾기
for index, line in enumerate(lines):
    if "참고 1" in line:
        print("참고자료 시작 위치:", index + 1, "행")
        print("해당 줄:", line)


참고자료 시작 위치: 77 행
해당 줄: 참고 1  정부포상 대상 명단(13명)


In [41]:

import re

# 기존 셀에서 확인한 95092531 보도자료
target_text = cleaned_df.loc[
    cleaned_df["보도자료ID"] == 95092531,
    "본문"
].iloc[0]

lines = target_text.splitlines()

print("===== 참고자료 제목 확인 =====")

found_count = 0

for index, line in enumerate(lines, start=1):
    # '참고 1', '참고 2' 등의 제목만 찾기
    if re.match(r"^\s*참고\s*\d+", line):
        print(f"{index}행: {line}")
        found_count += 1

print("\n참고자료 제목 발견 수:", found_count)


===== 참고자료 제목 확인 =====
77행: 참고 1  정부포상 대상 명단(13명)
228행: 참고 2  국토교통부 장관표창 대상 명단(45명)
421행: 참고 3 세미나 주제별 강연 내용

참고자료 제목 발견 수: 3


In [42]:
# 보도자료 95092531의 본문과 참고자료 경계 확인

target_text = cleaned_df.loc[
    cleaned_df["보도자료ID"] == 95092531,
    "본문"
].iloc[0]

lines = target_text.splitlines()

print("===== 본문과 참고자료 경계 =====")

for index, line in enumerate(lines, start=1):
    if 60 <= index <= 80:
        print(f"{index}행: {line}")

===== 본문과 참고자료 경계 =====
60행: 과  장 
61행: 손덕환(044-201-3993)
62행: <총괄>
63행: 물류정책과
64행: 담당자
65행: 사무관
66행: 이경섭(044-201-3996)
67행: 주무관
68행: 이수연(044-201-4000)
69행: 관련 기관
70행: 한국통합물류협회
71행: 상근부회장 
72행: 조무영(02-6954-6606)
73행: 팀장
74행: 박양호(02-6954-6613)
75행: 
76행: 
77행: 참고 1  정부포상 대상 명단(13명)
78행: 훈격
79행: 소속
80행: (직위)


In [43]:

# 보도자료 95092531의 본문 종료 부분 확인

target_text = cleaned_df.loc[
    cleaned_df["보도자료ID"] == 95092531,
    "본문"
].iloc[0]

lines = target_text.splitlines()

print("===== 기사 본문 종료 부분 =====")

for index in range(30, min(60, len(lines))):
    print(f"{index + 1}행: {lines[index]}")


===== 기사 본문 종료 부분 =====
31행: 연구위원에게 수여된다.
32행:  ㅇ 그 외 대통령표창은 ㈜광진티엘에스 김경호(대표), 와이에스엘에이전시(주) 
33행: 조동제(대표), 강원화물자동차운송주선사업협회 지동화(이사장), 육군종합보급
34행: 창에게, 국무총리표창은 ㈜태은물류 구은정(대표), ㈜윌로그 배성훈(대표), 
35행: 한국교통연구원 이지선(선임연구위원), ㈜센토인터내셔날 김정동(대표), 개인
36행: 화물운송사업자 김종득(화물운전자), ㈜삼영로지스틱스에게 수여된다.
37행:   - 이와 함께 물류현장에서 10년 이상 묵묵히 업무를 수행해온 택배기사(2명)와 
38행: 화물운전자(7명)를 비롯한 36명의 물류종사자와 9개 물류기업 및 단체에 
39행: 총 45점(세부명단 첨부)의 국토교통부 장관 표창도 수여한다.
40행: □ 2부에서는 미래물류기술의 산업환경 변화에 대한 대응 전략을 위해 ‘변화
41행: 하는 물류, 새로운 기회를 말하다’라는 대주제로 3개의 세미나(세부주제 
42행: 첨부)가 열린다.
43행:  ㅇ 100만 건 이상의 실제 운송 경험을 기반으로 구축된 AI Agent를 통해 
44행: 운송 전 과정의 데이터를 연결하고 실시간 최적화를 구현함으로써, 운영 
45행: 효율 개선을 넘어 물류 비즈니스의 확장 가능성을 모색하고,
46행:   - 기후위기로 새롭게 주목받는 북극항로의 실측 데이터를 바탕으로, 심화
47행: 되는 공급망‧기후‧지정학적 리스크를 대응하고 이를 극복할 수 있는 
48행: 방안을 공유하는 자리를 갖는다.
49행: □ 국토교통부 정의경 제2차관은 축사를 통해 “물류산업 매출액이 225조 규모로 
50행: 급성장할 수 있도록 산업현장에서 묵묵히 헌신해 오신 153만 물류인 
51행: 여러분께 깊은 감사를 드린다”면서,
52행:  ㅇ 정부도 “물류산업의 지속 발전을 견인하기 위해 첨단 물류 인프라 
53행: 확충과 함께 안전하고 지속가능한 근로여건 개선 마련을 위해 다양한 
54행: 지원과 정책적 노력을 펼칠 

In [44]:

# 보도자료 95092531의 본문과 참고자료 분리

target_id = 95092531

target_text = cleaned_df.loc[
    cleaned_df["보도자료ID"] == target_id,
    "본문"
].iloc[0]

lines = target_text.splitlines()

# 확인한 구간을 기준으로 분리
article_text = "\n".join(lines[:54]).strip()
reference_text = "\n".join(lines[76:]).strip()

print("===== 분리 결과 =====")
print("기사 본문 글자 수:", len(article_text))
print("참고자료 글자 수:", len(reference_text))

print("\n===== 기사 본문 전체 =====")
print(article_text)

print("\n===== 참고자료 전체 =====")
print(reference_text)


===== 분리 결과 =====
기사 본문 글자 수: 1872
참고자료 글자 수: 3615

===== 기사 본문 전체 =====
보도자료
보도시점 : 2026. 10. 28.(수) 11:00 이후(10. 29.(목) 조간) / 배포 : 2026. 10. 28.(수)
제34회 물류의 날 기념식
… 물류산업의 지속가능 발전방향 모색
- 30일 기념식에서 한국물류대상 총 58점 포상… 153만 물류인 자긍심 고취의 장 마련
□ 국토교통부(장관 홍지선)는 10월 30일 오전 대한상공회의소(국제회의장)
에서 개최되는 “제34회 물류의 날” 기념식*에서 그간 물류산업 발전에 
기여한 물류인 및 단체에게 한국물류대상을 수여한다.
    * (주최) 한국통합물류협회, (후원) 국토교통부, 대한상공회의소
 ㅇ 기념식은 물류산업 발전에 기여한 종사자들을 대상으로 열리는 한국물류
대상 시상식(1부, 본행사)에 이어 물류산업 육성에 필요한 최신 정보와 
지식을 공유하는 세미나(2부, 부대행사)로 나눠서 진행한다.
  - 기념식에는 정의경 국토교통부 제2차관을 비롯하여 신영수 한국통합
물류협회장 등 500여명의 물류업계 종사자들이 참석할 예정이다.
   * 유의동, 복기왕, 정동만 위원은 영상축사로 참여
□ 먼저, ‘한국물류대상 시상식’에서는 물류산업 발전에 기여한 기업인과 단체
에 훈장 1점, 산업포장 2점, 대통령표창 4점, 국무총리표창 6점, 그리고 
국토교통부 장관표창 45점 등 총 58점을 포상한다.
 ㅇ 최고의 영예인 은탑 산업훈장은 46년간 물류기업의 디지털 전환을 지원
하여 물류비 절감 등에 이바지해 온 ㈜한국비즈넷 김용필 대표이사에
게 수여된다.
  - 김용필 대표이사는 국내 최초로 국제화물 운송주선업 전용 정보시스템을 개발
하여 1,500여개 중소물류기업의 디지털 전환을 지원하는 등의 공로를 인정받았다.
 ㅇ 산업포장은 31년간 화물운송주선업에 종사하면서 수도권 콜드체인 인프라 
구축에 참여하는 등 국내 신선식품 물류체계 구축에 기여해 온 동원물류(주) 
신경용 

In [45]:

# 원본의 전체 줄 유지
original_lines = cleaned_df.loc[
    cleaned_df["보도자료ID"] == 95092531,
    "본문"
].iloc[0].splitlines()

# 앞서 확인한 세 영역
article_lines = original_lines[:54]
contact_lines = original_lines[54:76]
reference_lines = original_lines[76:]

# 세 영역을 다시 합치기
restored_lines = (
    article_lines + contact_lines + reference_lines
)

print("===== 분리 검증 =====")
print("원본 전체 줄 수:", len(original_lines))
print("기사 영역:", len(article_lines))
print("담당자 영역:", len(contact_lines))
print("참고자료 영역:", len(reference_lines))

print(
    "전체 줄 보존:",
    original_lines == restored_lines
)

print(
    "참고자료 시작:",
    reference_lines[0] if reference_lines else "없음"
)


===== 분리 검증 =====
원본 전체 줄 수: 506
기사 영역: 54
담당자 영역: 22
참고자료 영역: 430
전체 줄 보존: True
참고자료 시작: 참고 1  정부포상 대상 명단(13명)


In [46]:

# 분리된 기사 본문에 핵심 표현이 남아 있는지 확인

check_items = [
    "제34회 물류의 날",
    "한국물류대상",
    "58점",
    "153만 물류인",
    "미래물류기술",
]

print("===== 학습용 본문 핵심 표현 검사 =====")

for item in check_items:
    print(
        f"{item}:",
        "OK" if item in article_text else "확인 필요"
    )

print("\n기사 본문 글자 수:", len(article_text))


===== 학습용 본문 핵심 표현 검사 =====
제34회 물류의 날: OK
한국물류대상: OK
58점: OK
153만 물류인: OK
미래물류기술: OK

기사 본문 글자 수: 1872


In [47]:

import re
import pandas as pd

boundary_results = []

for _, row in cleaned_df.iterrows():
    news_id = row["보도자료ID"]

    # 이미 개별적으로 확인한 보도자료는 제외
    if news_id == 95092531:
        continue

    lines = row["본문"].splitlines()

    # 담당부서 표기가 있는 줄 검색
    positions = [
        index
        for index, line in enumerate(lines)
        if re.search(r"담당\s*부서", line)
    ]

    # 발견된 위치와 직전 줄 확인
    if positions:
        position = positions[-1]

        boundary_results.append({
            "보도자료ID": news_id,
            "전체줄수": len(lines),
            "담당부서위치": position + 1,
            "직전줄": lines[position - 1] if position > 0 else "",
            "해당줄": lines[position],
        })
    else:
        boundary_results.append({
            "보도자료ID": news_id,
            "전체줄수": len(lines),
            "담당부서위치": None,
            "직전줄": "",
            "해당줄": "표기 없음",
        })

boundary_df = pd.DataFrame(boundary_results)

display(boundary_df)


,보도자료ID,전체줄수,담당부서위치,직전줄,해당줄
0,95092576,58,45.0,절차가 지연되지 않도록 각별히 노력해 달라”고 강조하였다.,담당부서
1,95092592,73,NaN,,표기 없음
2,95092593,39,32.0,적극적인 협조”를 당부했다.,담당 부서
3,95092583,187,53.0,,담당 부서
4,95092574,108,70.0,고 밝혔다.,담당 부서
5,95092564,49,42.0,적극 해결해 달라”고 당부하였다.,담당 부서
6,95092547,59,51.0,정유진(044-203-2873),담당 부서
7,95092532,56,49.0,밝힐 예정이다.,담당 부서
8,95092519,56,43.0,`,담당부서


In [48]:

import re

# 담당자 정보 구조가 다른 보도자료 2건
target_ids = [95092592, 95092547]

for news_id in target_ids:
    row = cleaned_df.loc[
        cleaned_df["보도자료ID"] == news_id
    ].iloc[0]

    lines = row["본문"].splitlines()

    print("\n===== 보도자료 확인 =====")
    print("ID:", news_id)
    print("제목:", row["제목"])
    print("전체 줄 수:", len(lines))

    # 담당자 정보와 관련된 줄 찾기
    for index, line in enumerate(lines):
        if re.search(r"담당\s*부서|담당자|책임자|044-\d+", line):
            print(f"{index + 1}행: {line}")



===== 보도자료 확인 =====
ID: 95092592
제목: 물류기업과 기술기업이 만나는  ‘협업의 장’ 열린다
전체 줄 수: 73
56행: 책임자
58행: 손덕환(044-201-3993)
59행: 담당자
61행: 박승연(044-201-3995)

===== 보도자료 확인 =====
ID: 95092547
제목: 문체부-국토부, 관광교통 칸막이 허물고 ‘지방관광 시대’ 연다.
전체 줄 수: 59
42행: 담당 부서
44행: 책임자
46행: 김진희(044-203-2870)
48행: 담당자
50행: 정유진(044-203-2873)
51행: 담당 부서
53행: 책임자
55행: 안광열(044-201-3804)
57행: 담당자
59행: 류나린(044-201-3791)


In [49]:

# 두 보도자료의 본문과 담당자 정보 경계 확인
boundary_targets = {
    95092592: 56,
    95092547: 42,
}

for news_id, contact_start in boundary_targets.items():
    text = cleaned_df.loc[
        cleaned_df["보도자료ID"] == news_id,
        "본문"
    ].iloc[0]

    lines = text.splitlines()

    print(f"\n===== 보도자료 {news_id} =====")

    # 담당자 영역 시작 전후의 문장 확인
    for index in range(
        max(0, contact_start - 6),
        min(len(lines), contact_start + 3)
    ):
        print(f"{index + 1}행: {lines[index]}")



===== 보도자료 95092592 =====
51행: 기대한다”라며, 
52행:  ㅇ “국토교통부는 앞으로도 물류산업의 기술 혁신과 기업 간 상생협력을 
53행: 위한 정책을 지속적으로 추진하겠다”라고 밝혔다. 
54행: 국토교통부
55행: 물류정책과
56행: 책임자
57행: 과  장
58행: 손덕환(044-201-3993)
59행: 담당자

===== 보도자료 95092547 =====
37행:   국토부 박재순 교통물류실장은 "외래객이 지방공항으로 더 많이 들어올 
38행: 수 있도록 국제공항 슬롯 확대와 인바운드 항공편 우선 배정 등 국제선을 
39행: 늘려 하늘길을 넓히고, 공항부터 지역까지 편리하게 이동할 수 있도록 
40행: 철도·버스·택시 서비스와 결제 환경을 개선하겠다. 문체부와의 긴밀한 
41행: 협력을 통해 지방공항이 외래객의 새로운 관문이 되도록 하겠다."라고 밝혔다.
42행: 담당 부서
43행: 문화체육관광부
44행: 책임자
45행: 과장


In [50]:

# 보도자료 95092592의 담당자 정보 이후 전체 내용 확인

target_text = cleaned_df.loc[
    cleaned_df["보도자료ID"] == 95092592,
    "본문"
].iloc[0]

lines = target_text.splitlines()

print("===== 담당자 정보 이후 내용 =====")

for index in range(53, len(lines)):
    print(f"{index + 1}행: {lines[index]}")


===== 담당자 정보 이후 내용 =====
54행: 국토교통부
55행: 물류정책과
56행: 책임자
57행: 과  장
58행: 손덕환(044-201-3993)
59행: 담당자
60행: 사무관
61행: 박승연(044-201-3995)
62행: 
63행: 
64행: 참고1
65행:  물류상생 Meet-up 행사 포스터
66행: 
67행: 
68행: 참고2
69행:  참여기업 소개
70행: 
71행: 
72행: 
73행: 


In [51]:

# 원본과 대조해 확인한 본문 종료 위치
article_end_positions = {
    95092592: 53,
    95092547: 41,
}

for news_id, end_line in article_end_positions.items():
    row = cleaned_df.loc[
        cleaned_df["보도자료ID"] == news_id
    ].iloc[0]

    lines = row["본문"].splitlines()

    # 원본에서 확인한 본문 영역만 분리
    article_lines = lines[:end_line]
    separated_article = "\n".join(article_lines).strip()

    print(f"\n===== 보도자료 {news_id} =====")
    print("본문 줄 수:", len(article_lines))
    print("본문 글자 수:", len(separated_article))

    print("\n마지막 문장:")
    print(article_lines[-1])

    print("\n본문 전체:")
    print(separated_article)



===== 보도자료 95092592 =====
본문 줄 수: 53
본문 글자 수: 1700

마지막 문장:
위한 정책을 지속적으로 추진하겠다”라고 밝혔다. 

본문 전체:
보도자료 
 
보도시점 : 2026. 11. 13.(금) 06:00 이후(11. 13.(금) 석간) / 배포 : 2026. 11. 12.(목)
물류기업과 기술기업이 만나는 
‘협업의 장’ 열린다
- 11월 13일 서울창업허브 공덕에서 “2026 물류상생 Meet-up” 개최
- 우수기술 발표·개방형 혁신(OI) 사례 공유·1:1 협업 매칭까지 한자리에  
□ 국토교통부(장관 홍지선)와 물류산업진흥재단(이사장 심재선)은 11월 
13일 오후 서울창업허브 공덕에서 “2026 물류상생 Meet-up”을 개최한다.
 ㅇ 물류산업의 패러다임 전환에 따라 물류 기술 도입 수요는 꾸준히 
증가하고 있으나, 물류-기술 기업 간의 협업 기회가 부족해 우수한 
신기술이 실제 현장에 적용되는 데에는 한계가 있었다. 
 ㅇ 이에 국토교통부는 정부 주도로 물류-기술기업 파트너십 구축 계기를 
마련하고, 협업을 통한 물류산업 고도화를 지원하고자 물류산업진흥
재단과 함께 “2026 물류상생 Meet-up”을 개최하게 되었다.
□ 물류상생 Meet-up에서는 기술기업이 보유하고 있는 우수 기술을 소개
하는 한편, 행사에 참여한 물류기업(수요기업)과 기술도입 및 실증을 위한 
협업 방안에 대하여 1:1로 논의할 예정이다. 
 ㅇ 아울러, 국토교통부는 우수 기술기업과 물류기업 간 협업사업이 실질
적인 사업 성과로 이어질 수 있도록 정부 지원사업과 연계해 기업 간 
협력 기반을 마련하고, 기술 도입과 현장 적용을 적극 지원할 계획이다. 
□ 이번 행사에서는 기술도입․실증을 위한 협업 파트너십 구축 성사율을 
높이기 위해 참여 물류기업(10개사)을 대상으로 기술기업과 협업을 
희망하는 8개 대과제를 사전 조사하였다. 이를 토대로 해당 기술을 
보유한 기업을 대상으로 참여 기술기업(23개사)을 선정하였다.
 
 < Meet-u

In [52]:

import re

# 앞서 확인한 본문 종료 위치
article_end_positions = {
    95092592: 53,
    95092547: 41,
}

phone_pattern = r"0\d{1,2}[- ]\d{3,4}[- ]\d{4}"

for news_id, end_line in article_end_positions.items():
    row = cleaned_df.loc[
        cleaned_df["보도자료ID"] == news_id
    ].iloc[0]

    original_lines = row["본문"].splitlines()

    # 원본에서 확인한 본문 영역 분리
    article_lines = original_lines[:end_line]
    article_text_check = "\n".join(article_lines).strip()

    print(f"\n===== 보도자료 {news_id} =====")
    print("본문 줄 수:", len(article_lines))
    print("본문 글자 수:", len(article_text_check))
    print("전화번호 포함:", bool(re.search(phone_pattern, article_text_check)))
    print("마지막 줄:", article_lines[-1])
    print("원본 줄 보존:", article_lines == original_lines[:end_line])



===== 보도자료 95092592 =====
본문 줄 수: 53
본문 글자 수: 1700
전화번호 포함: False
마지막 줄: 위한 정책을 지속적으로 추진하겠다”라고 밝혔다. 
원본 줄 보존: True

===== 보도자료 95092547 =====
본문 줄 수: 41
본문 글자 수: 1304
전화번호 포함: False
마지막 줄: 협력을 통해 지방공항이 외래객의 새로운 관문이 되도록 하겠다."라고 밝혔다.
원본 줄 보존: True


In [53]:

import re
import pandas as pd

# 본문 경계를 확인한 3건 제외
checked_ids = {95092531, 95092592, 95092547}

boundary_results = []

for _, row in cleaned_df.iterrows():
    news_id = int(row["보도자료ID"])

    if news_id in checked_ids:
        continue

    lines = row["본문"].splitlines()

    # 담당자 정보 영역으로 추정되는 위치 탐색
    positions = [
        i for i, line in enumerate(lines)
        if re.fullmatch(r"\s*담당\s*부서\s*", line)
    ]

    if positions:
        start = positions[-1]

        # 경계 직전의 빈 줄과 구분선은 건너뛰기
        before_lines = [
            line.strip()
            for line in lines[:start]
            if line.strip() and line.strip() not in {"-", "□"}
        ]

        last_line = before_lines[-1] if before_lines else ""

        boundary_results.append({
            "보도자료ID": news_id,
            "전체줄수": len(lines),
            "담당부서시작": start + 1,
            "직전내용": last_line,
            "검토상태": "경계 확인 필요"
        })
    else:
        boundary_results.append({
            "보도자료ID": news_id,
            "전체줄수": len(lines),
            "담당부서시작": None,
            "직전내용": "",
            "검토상태": "수동 확인 필요"
        })

boundary_review_df = pd.DataFrame(boundary_results)

display(boundary_review_df)


,보도자료ID,전체줄수,담당부서시작,직전내용,검토상태
0,95092576,58,45,절차가 지연되지 않도록 각별히 노력해 달라”고 강조하였다.,경계 확인 필요
1,95092593,39,32,적극적인 협조”를 당부했다.,경계 확인 필요
2,95092583,187,53,해 달라”고 강조하였다.,경계 확인 필요
3,95092574,108,70,고 밝혔다.,경계 확인 필요
4,95092564,49,42,적극 해결해 달라”고 당부하였다.,경계 확인 필요
5,95092532,56,49,밝힐 예정이다.,경계 확인 필요
6,95092519,56,43,`,경계 확인 필요


In [54]:

# 담당자 영역이 시작되는 위치 앞뒤 확인

for _, row in boundary_review_df.iterrows():
    news_id = int(row["보도자료ID"])
    start_line = int(row["담당부서시작"])

    text = cleaned_df.loc[
        cleaned_df["보도자료ID"] == news_id,
        "본문"
    ].iloc[0]

    lines = text.splitlines()

    print(f"\n===== 보도자료 {news_id} =====")
    print("담당자 영역 시작:", start_line, "행")

    # 경계 앞뒤 3줄 확인
    for index in range(
        max(0, start_line - 4),
        min(len(lines), start_line + 3)
    ):
        print(f"{index + 1}행: {lines[index]}")



===== 보도자료 95092576 =====
담당자 영역 시작: 45 행
42행: 기관에서는 사업별 추진일정을 철저히 관리하고, 관계기관 간 협의가 
43행: 필요한 사항은 대광위와 긴밀히 공유하고 선제적으로 조정하여 후속 
44행: 절차가 지연되지 않도록 각별히 노력해 달라”고 강조하였다.
45행: 담당부서
46행: 대도시권광역교통위원회
47행: 광역교통정책과
48행: 책임자

===== 보도자료 95092593 =====
담당자 영역 시작: 32 행
29행: 추진상황과 현장 여건을 직접 확인하면서, 
30행: ㅇ “사업이 계획된 일정에 맞춰 차질 없이 추진될 수 있도록 관계기관의 
31행: 적극적인 협조”를 당부했다.
32행: 담당 부서
33행: 대도시권광역교통위원회
34행: 광역교통정책과
35행: 책임자

===== 보도자료 95092583 =====
담당자 영역 시작: 53 행
50행: 해 달라”고 강조하였다.
51행: 
52행: 
53행: 담당 부서
54행: 교통정책총괄과
55행: 책임자
56행: 과 장

===== 보도자료 95092574 =====
담당자 영역 시작: 70 행
67행:  ㅇ “시에서는 어르신들이 새로 출시되는 카드를 쉽게 발급받을 수 있도록 
68행: 관계기관과 현장 홍보를 실시하고 이용방법 안내를 적극 지원하겠다.”
69행: 고 밝혔다.
70행: 담당 부서
71행: 국토교통부
72행: 광역교통경제과
73행: 책임자

===== 보도자료 95092564 =====
담당자 영역 시작: 42 행
39행: 교통시설이 적기에 구축될 수 있도록 사업별 공정관리를 철저히 하고, 
40행: 사업 추진 과정에서 발생하는 현안도 관계기관 간 긴밀한 협의를 통해 
41행: 적극 해결해 달라”고 당부하였다.
42행: 담당 부서
43행: 대도시권광역교통위원회
44행: 광역교통정책과
45행: 책임자

===== 보도자료 95092532 =====
담당자 영역 시작: 49 행
46행: 대책이 차질 없이 이행될 수 있도록 대광위가 적극적인 조정 역할을 수

In [55]:

import pandas as pd

boundary_check_results = []

for _, row in boundary_review_df.iterrows():
    news_id = int(row["보도자료ID"])
    start_line = int(row["담당부서시작"])

    text = cleaned_df.loc[
        cleaned_df["보도자료ID"] == news_id,
        "본문"
    ].iloc[0]

    lines = text.splitlines()

    # 담당자 영역 바로 앞과 시작 부분 확인
    boundary_check_results.append({
        "보도자료ID": news_id,
        "담당부서시작": start_line,
        "시작전_3행": lines[start_line - 4],
        "시작전_2행": lines[start_line - 3],
        "시작전_1행": lines[start_line - 2],
        "시작행": lines[start_line - 1],
        "시작다음행": lines[start_line]
        if start_line < len(lines) else ""
    })

boundary_check_df = pd.DataFrame(boundary_check_results)

# 모든 행과 열을 생략하지 않고 표로 표시
with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None
):
    display(boundary_check_df)


,보도자료ID,담당부서시작,시작전_3행,시작전_2행,시작전_1행,시작행,시작다음행
0,95092576,45,"기관에서는 사업별 추진일정을 철저히 관리하고, 관계기관 간 협의가",필요한 사항은 대광위와 긴밀히 공유하고 선제적으로 조정하여 후속,절차가 지연되지 않도록 각별히 노력해 달라”고 강조하였다.,담당부서,대도시권광역교통위원회
1,95092593,32,"추진상황과 현장 여건을 직접 확인하면서,",ㅇ “사업이 계획된 일정에 맞춰 차질 없이 추진될 수 있도록 관계기관의,적극적인 협조”를 당부했다.,담당 부서,대도시권광역교통위원회
2,95092583,53,해 달라”고 강조하였다.,,,담당 부서,교통정책총괄과
3,95092574,70,ㅇ “시에서는 어르신들이 새로 출시되는 카드를 쉽게 발급받을 수 있도록,관계기관과 현장 홍보를 실시하고 이용방법 안내를 적극 지원하겠다.”,고 밝혔다.,담당 부서,국토교통부
4,95092564,42,"교통시설이 적기에 구축될 수 있도록 사업별 공정관리를 철저히 하고,",사업 추진 과정에서 발생하는 현안도 관계기관 간 긴밀한 협의를 통해,적극 해결해 달라”고 당부하였다.,담당 부서,대도시권광역교통위원회
5,95092532,49,대책이 차질 없이 이행될 수 있도록 대광위가 적극적인 조정 역할을 수,"행하고, 신속추진 TF를 통해 주요 사업의 걸림돌을 해소해 나가겠다”라고",밝힐 예정이다.,담당 부서,대도시권광역교통위원회
6,95092519,43,"구축될 수 있도록 사업별 추진상황을 면밀히 관리하고, ‘직접 인허가’ 등",가용한 제도를 적극 활용하는 등 모든 역량을 집중하겠다“고 밝혔다.,`,담당부서,대도시권광역교통위원회


In [56]:

import re
import pandas as pd

# 앞서 확인한 담당자 영역 시작 위치
contact_start_lines = {
    95092576: 45,
    95092593: 32,
    95092583: 53,
    95092574: 70,
    95092564: 42,
    95092532: 49,
    95092519: 43,
}

separated_results = []

phone_pattern = r"0\d{1,2}[- ]\d{3,4}[- ]\d{4}"

for news_id, start_line in contact_start_lines.items():
    row = cleaned_df.loc[
        cleaned_df["보도자료ID"] == news_id
    ].iloc[0]

    original_lines = row["본문"].splitlines()

    # 담당부서 시작 직전까지 본문으로 분리
    article_lines = original_lines[:start_line - 1]
    article_text = "\n".join(article_lines).strip()

    separated_results.append({
        "보도자료ID": news_id,
        "본문줄수": len(article_lines),
        "본문글자수": len(article_text),
        "전화번호포함": bool(
            re.search(phone_pattern, article_text)
        ),
        "마지막줄": article_lines[-1] if article_lines else "",
    })

separated_df = pd.DataFrame(separated_results)

display(separated_df)


,보도자료ID,본문줄수,본문글자수,전화번호포함,마지막줄
0,95092576,44,1592,False,절차가 지연되지 않도록 각별히 노력해 달라”고 강조하였다.
1,95092593,31,925,False,적극적인 협조”를 당부했다.
2,95092583,52,1675,False,
3,95092574,69,2389,False,고 밝혔다.
4,95092564,41,1281,False,적극 해결해 달라”고 당부하였다.
5,95092532,48,1521,False,밝힐 예정이다.
6,95092519,42,1397,False,`


In [57]:

# 분리한 7건의 본문 마지막 3줄 확인

for news_id, start_line in contact_start_lines.items():
    row = cleaned_df.loc[
        cleaned_df["보도자료ID"] == news_id
    ].iloc[0]

    lines = row["본문"].splitlines()
    article_lines = lines[:start_line - 1]

    print(f"\n===== 보도자료 {news_id} =====")

    for line in article_lines[-3:]:
        print(repr(line))



===== 보도자료 95092576 =====
'기관에서는 사업별 추진일정을 철저히 관리하고, 관계기관 간 협의가 '
'필요한 사항은 대광위와 긴밀히 공유하고 선제적으로 조정하여 후속 '
'절차가 지연되지 않도록 각별히 노력해 달라”고 강조하였다.'

===== 보도자료 95092593 =====
'추진상황과 현장 여건을 직접 확인하면서, '
'ㅇ “사업이 계획된 일정에 맞춰 차질 없이 추진될 수 있도록 관계기관의 '
'적극적인 협조”를 당부했다.'

===== 보도자료 95092583 =====
'해 달라”고 강조하였다.'
''
''

===== 보도자료 95092574 =====
' ㅇ “시에서는 어르신들이 새로 출시되는 카드를 쉽게 발급받을 수 있도록 '
'관계기관과 현장 홍보를 실시하고 이용방법 안내를 적극 지원하겠다.”'
'고 밝혔다.'

===== 보도자료 95092564 =====
'교통시설이 적기에 구축될 수 있도록 사업별 공정관리를 철저히 하고, '
'사업 추진 과정에서 발생하는 현안도 관계기관 간 긴밀한 협의를 통해 '
'적극 해결해 달라”고 당부하였다.'

===== 보도자료 95092532 =====
'대책이 차질 없이 이행될 수 있도록 대광위가 적극적인 조정 역할을 수'
'행하고, 신속추진 TF를 통해 주요 사업의 걸림돌을 해소해 나가겠다”라고 '
'밝힐 예정이다.'

===== 보도자료 95092519 =====
'구축될 수 있도록 사업별 추진상황을 면밀히 관리하고, ‘직접 인허가’ 등 '
'가용한 제도를 적극 활용하는 등 모든 역량을 집중하겠다“고 밝혔다.'
'`'


In [58]:

# 본문 끝에 문제가 의심되는 보도자료 2건
target_ids = [95092583, 95092519]

for news_id in target_ids:
    start_line = contact_start_lines[news_id]

    row = cleaned_df.loc[
        cleaned_df["보도자료ID"] == news_id
    ].iloc[0]

    lines = row["본문"].splitlines()
    article_lines = lines[:start_line - 1]

    print(f"\n===== 보도자료 {news_id} =====")

    # 끝부분의 실제 문자 확인
    for line in article_lines[-5:]:
        print("내용:", repr(line))

    # 마지막 줄의 유니코드 문자 코드 확인
    last_line = article_lines[-1]

    print("마지막 줄 문자 코드:")
    print([f"U+{ord(char):04X}" for char in last_line])



===== 보도자료 95092583 =====
내용: ' ㅇ 정 차관은 현장 관계자들에게 “공사 기간 단축도 중요하지만 안전이 '
내용: '최우선”이라며, “철저한 안전관리를 바탕으로 공사를 차질 없이 진행'
내용: '해 달라”고 강조하였다.'
내용: ''
내용: ''
마지막 줄 문자 코드:
[]

===== 보도자료 95092519 =====
내용: '겪지 않도록 광역교통개선대책을 속도감 있게 추진해야 한다”면서,'
내용: ' ㅇ 앞으로도 3기 신도시의 주택공급 일정에 맞춰 광역교통시설이 적기에 '
내용: '구축될 수 있도록 사업별 추진상황을 면밀히 관리하고, ‘직접 인허가’ 등 '
내용: '가용한 제도를 적극 활용하는 등 모든 역량을 집중하겠다“고 밝혔다.'
내용: '`'
마지막 줄 문자 코드:
['U+0060']


In [59]:

import re
import pandas as pd

# 지금까지 개별적으로 확인한 기사 본문 종료 줄
# 줄 번호는 기존 cleaned_df의 본문을 기준으로 함
article_end_lines = {
    95092576: 44,
    95092592: 53,
    95092593: 31,
    95092583: 52,
    95092574: 69,
    95092564: 41,
    95092547: 41,
    95092532: 48,
    95092531: 54,
    95092519: 42,
}

# 원본 데이터는 보존하고 복사본에서 작업
training_df = cleaned_df.copy()

# 검증되지 않은 ID가 섞여 있으면 처리 중단
data_ids = set(training_df["보도자료ID"].astype(int))

if data_ids != set(article_end_lines):
    raise ValueError("보도자료 ID가 기존 검증 결과와 다릅니다.")

def extract_article_body(row):
    news_id = int(row["보도자료ID"])
    end_line = article_end_lines[news_id]

    # 원문에서 확인한 기사 영역만 분리
    lines = row["본문"].splitlines()

    if len(lines) < end_line:
        raise ValueError(f"{news_id}: 본문 줄 수가 부족합니다.")

    article_lines = lines[:end_line]

    # 본문 마지막의 빈 줄 또는 단독 백틱 제거
    while article_lines and article_lines[-1].strip() in ("", "`"):
        article_lines.pop()

    return "\n".join(article_lines).strip()

# 학습용 본문을 별도 열에 저장
training_df["학습본문"] = training_df.apply(
    extract_article_body,
    axis=1
)

# 검증 정보 계산
phone_pattern = r"0\d{1,2}[- ]\d{3,4}[- ]\d{4}"

training_df["학습본문글자수"] = training_df["학습본문"].str.len()

training_df["전화번호포함"] = training_df["학습본문"].apply(
    lambda text: bool(re.search(phone_pattern, text))
)

training_df["본문누락"] = training_df["학습본문"].str.strip().eq("")

training_df["마지막줄"] = training_df["학습본문"].apply(
    lambda text: text.splitlines()[-1] if text else ""
)

print("===== 통합 정제 결과 =====")
print("전체 보도자료:", len(training_df))
print("학습본문 누락:", training_df["본문누락"].sum())
print("전화번호 포함:", training_df["전화번호포함"].sum())
print("학습본문 전체 글자 수:", training_df["학습본문글자수"].sum())

display(
    training_df[
        ["보도자료ID", "학습본문글자수",
         "전화번호포함", "본문누락", "마지막줄"]
    ]
)


===== 통합 정제 결과 =====
전체 보도자료: 10
학습본문 누락: 0
전화번호 포함: 0
학습본문 전체 글자 수: 15654


,보도자료ID,학습본문글자수,전화번호포함,본문누락,마지막줄
0,95092576,1592,False,False,절차가 지연되지 않도록 각별히 노력해 달라”고 강조하였다.
1,95092592,1700,False,False,위한 정책을 지속적으로 추진하겠다”라고 밝혔다.
2,95092593,925,False,False,적극적인 협조”를 당부했다.
3,95092583,1675,False,False,해 달라”고 강조하였다.
4,95092574,2389,False,False,고 밝혔다.
5,95092564,1281,False,False,적극 해결해 달라”고 당부하였다.
6,95092547,1304,False,False,"협력을 통해 지방공항이 외래객의 새로운 관문이 되도록 하겠다.""라고 밝혔다."
7,95092532,1521,False,False,밝힐 예정이다.
8,95092531,1872,False,False,지원과 정책적 노력을 펼칠 계획”이라고 밝힐 예정이다.
9,95092519,1395,False,False,가용한 제도를 적극 활용하는 등 모든 역량을 집중하겠다“고 밝혔다.


In [60]:

from pathlib import Path
import pandas as pd

# 정제 데이터 저장 경로
output_file = Path("trafficnews_all_10_cleaned.csv")

# 학습 데이터에 필요한 열만 선택
columns = [
    "보도자료ID",
    "제목",
    "담당부서",
    "등록일",
    "상세URL",
    "PDF_URL",
    "학습본문"
]

result_df = training_df[columns].copy()

# 데이터 검증
if len(result_df) != 10:
    raise ValueError("데이터 수가 10건이 아닙니다.")

if result_df["학습본문"].isna().any():
    raise ValueError("학습 본문에 누락이 있습니다.")

if result_df["보도자료ID"].duplicated().any():
    raise ValueError("중복 보도자료 ID가 있습니다.")

# 기존 파일 덮어쓰기 방지
if output_file.exists():
    print("파일이 이미 존재합니다. 덮어쓰지 않았습니다.")
else:
    result_df.to_csv(
        output_file,
        index=False,
        encoding="utf-8-sig"
    )
    print("정제 CSV 저장 완료!")

# 저장 결과 확인
saved_df = pd.read_csv(output_file)

print("저장 건수:", len(saved_df))
print("본문 누락:", saved_df["학습본문"].isna().sum())
print(
    "전체 본문 일치:",
    saved_df["학습본문"].tolist() == result_df["학습본문"].tolist()
)
print("저장 파일:", output_file)


정제 CSV 저장 완료!
저장 건수: 10
본문 누락: 0
전체 본문 일치: True
저장 파일: trafficnews_all_10_cleaned.csv


In [61]:
import requests
from bs4 import BeautifulSoup

# 기존 목록 URL의 페이지 번호만 변경
page_url = (
    "https://www.molit.go.kr/USR/NEWS/m_71/lst.jsp"
    "?lcmspage=2&search_section=p_sec_5"
)

response = requests.get(page_url, timeout=20)
response.raise_for_status()
response.encoding = response.apparent_encoding

soup_page2 = BeautifulSoup(response.text, "html.parser")

# 상세 페이지 링크 수집
page2_links = []

for link in soup_page2.select("a[href]"):
    href = link.get("href", "")
    title = link.get_text(" ", strip=True)

    if "dtl.jsp" in href and title:
        page2_links.append((title, href))

print("HTTP 상태:", response.status_code)
print("2페이지 보도자료 수:", len(page2_links))

for title, href in page2_links:
    print("제목:", title)
    print("링크:", href)
    print()

HTTP 상태: 200
2페이지 보도자료 수: 10
제목: 3기 신도시 교통대책  ‘직접 인·허가’로 속도 높인다
링크: dtl.jsp?lcmspage=2&id=95092519

제목: 모두의카드 카드정보 현행화하지 않은 이용자는 10월 26일까지 현행화하세요!
링크: dtl.jsp?lcmspage=2&id=95092504

제목: 택시업종 노동환경 실태, 노 · 사 · 정이 함께 살핀다
링크: dtl.jsp?lcmspage=2&id=95092486

제목: 모두의카드 출퇴근 시차시간 혜택 계속 이어간다
링크: dtl.jsp?lcmspage=2&id=95092459

제목: [장관동정] 추석 귀성길 안전·편의 빈틈없도록 … 주요 교통현장 집중 점검
링크: dtl.jsp?lcmspage=2&id=95092458

제목: APEC 교통장관회의서 AI·스마트 모빌리티 미래협력 제안 나선다
링크: dtl.jsp?lcmspage=2&id=95092447

제목: 국토부·기후부, 두 마리 토끼 잡는다... 모두의카드·그린카드 연계
링크: dtl.jsp?lcmspage=2&id=95092445

제목: 추석 연휴 교통혼잡 해소에 총력... 정부 합동 특별교통대책을 시행합니다
링크: dtl.jsp?lcmspage=2&id=95092439

제목: 안전운전 배달 종사자 보험료   최대 11% 할인
링크: dtl.jsp?lcmspage=2&id=95092438

제목: 5극3특 국가균형발전 뒷받침할 초광역교통 전략 연내 마련
링크: dtl.jsp?lcmspage=2&id=95092430



In [62]:

import pandas as pd
from urllib.parse import urljoin, urlparse, parse_qs

# 2페이지 목록을 데이터프레임으로 변환
page2_data = []

for title, href in page2_links:
    detail_url = urljoin(page_url, href)
    news_id = parse_qs(
        urlparse(detail_url).query
    ).get("id", [""])[0]

    page2_data.append({
        "보도자료ID": news_id,
        "제목": title,
        "상세URL": detail_url,
    })

page2_df = pd.DataFrame(page2_data)

# 기존에 수집한 10건의 ID
existing_ids = set(
    pd.read_csv("trafficnews_all_10_raw.csv")[
        "보도자료ID"
    ].astype(str)
)

# 신규 보도자료만 선별
page2_new_df = page2_df[
    ~page2_df["보도자료ID"].isin(existing_ids)
].copy()

page2_new_df = page2_new_df.drop_duplicates(
    subset=["보도자료ID"]
).reset_index(drop=True)

print("2페이지 목록:", len(page2_df))
print("기존 수집 데이터와 중복:", len(page2_df) - len(page2_new_df))
print("신규 수집 대상:", len(page2_new_df))

display(page2_new_df)


2페이지 목록: 10
기존 수집 데이터와 중복: 1
신규 수집 대상: 9


,보도자료ID,제목,상세URL
0,95092504,모두의카드 카드정보 현행화하지 않은 이용자는 10월 26일까지 현행화하세요!,https://www.molit.go.kr/USR/NEWS/m_71/dtl.jsp?...
1,95092486,"택시업종 노동환경 실태, 노 · 사 · 정이 함께 살핀다",https://www.molit.go.kr/USR/NEWS/m_71/dtl.jsp?...
2,95092459,모두의카드 출퇴근 시차시간 혜택 계속 이어간다,https://www.molit.go.kr/USR/NEWS/m_71/dtl.jsp?...
3,95092458,[장관동정] 추석 귀성길 안전·편의 빈틈없도록 … 주요 교통현장 집중 점검,https://www.molit.go.kr/USR/NEWS/m_71/dtl.jsp?...
4,95092447,APEC 교통장관회의서 AI·스마트 모빌리티 미래협력 제안 나선다,https://www.molit.go.kr/USR/NEWS/m_71/dtl.jsp?...
5,95092445,"국토부·기후부, 두 마리 토끼 잡는다... 모두의카드·그린카드 연계",https://www.molit.go.kr/USR/NEWS/m_71/dtl.jsp?...
6,95092439,추석 연휴 교통혼잡 해소에 총력... 정부 합동 특별교통대책을 시행합니다,https://www.molit.go.kr/USR/NEWS/m_71/dtl.jsp?...
7,95092438,안전운전 배달 종사자 보험료 최대 11% 할인,https://www.molit.go.kr/USR/NEWS/m_71/dtl.jsp?...
8,95092430,5극3특 국가균형발전 뒷받침할 초광역교통 전략 연내 마련,https://www.molit.go.kr/USR/NEWS/m_71/dtl.jsp?...


In [63]:

import time
import requests
import pymupdf
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

new_records = []
failed_records = []

for _, row in page2_new_df.iterrows():
    news_id = row["보도자료ID"]
    detail_url = row["상세URL"]

    try:
        # 상세 페이지 가져오기
        response = requests.get(detail_url, timeout=20)
        response.raise_for_status()
        response.encoding = response.apparent_encoding

        soup = BeautifulSoup(response.text, "html.parser")

        # 제목 확인
        title_tag = soup.select_one("h4")
        title = title_tag.get_text(" ", strip=True) if title_tag else ""

        department = ""
        published_at = ""

        for item in soup.select("ul.bd_view_ul_info li"):
            label = item.select_one("strong")
            value = item.select_one("span")

            if label is None or value is None:
                continue

            key = label.get_text(strip=True)
            text = value.get_text(" ", strip=True)

            if key == "담당부서":
                department = text
            elif key == "등록일":
                published_at = text

        # PDF 다운로드 URL 찾기
        pdf_url = ""

        for link in soup.select("li.file a[href]"):
            file_name = link.get_text(" ", strip=True)

            if ".pdf" in file_name.lower():
                pdf_url = urljoin(detail_url, link.get("href"))
                break

        if not pdf_url:
            raise ValueError("PDF 첨부파일 없음")

        # 상세 페이지 요청 후 잠시 대기
        time.sleep(1)

        # PDF 전체 다운로드 및 텍스트 추출
        pdf_response = requests.get(pdf_url, timeout=30)
        pdf_response.raise_for_status()

        if not pdf_response.content.startswith(b"%PDF-"):
            raise ValueError("PDF 파일 형식 오류")

        with pymupdf.open(
            stream=pdf_response.content,
            filetype="pdf"
        ) as document:
            page_count = len(document)

            full_text = "\n".join(
                page.get_text("text")
                for page in document
            )

        if not full_text.strip():
            raise ValueError("본문 추출 결과가 비어 있음")

        new_records.append({
            "보도자료ID": news_id,
            "제목": title,
            "담당부서": department,
            "등록일": published_at,
            "상세URL": detail_url,
            "PDF_URL": pdf_url,
            "본문": full_text,
            "페이지수": page_count,
            "본문글자수": len(full_text),
        })

        print("수집 성공:", news_id, title)

    except (requests.RequestException, pymupdf.FileDataError, ValueError) as error:
        failed_records.append({
            "보도자료ID": news_id,
            "오류": str(error)
        })
        print("수집 실패:", news_id, error)

    # 다음 보도자료 요청 전 대기
    time.sleep(1)

# 결과 데이터프레임
page2_extracted_df = pd.DataFrame(new_records)

print("\n===== 신규 수집 결과 =====")
print("수집 성공:", len(new_records))
print("수집 실패:", len(failed_records))

if not page2_extracted_df.empty:
    display(
        page2_extracted_df[
            ["보도자료ID", "제목", "페이지수", "본문글자수"]
        ]
    )

if failed_records:
    display(pd.DataFrame(failed_records))


수집 성공: 95092504 모두의카드 카드정보 현행화하지 않은 이용자는 10월 26일까지 현행화하세요!
수집 성공: 95092486 택시업종 노동환경 실태, 노 · 사 · 정이 함께 살핀다
수집 성공: 95092459 모두의카드 출퇴근 시차시간 혜택 계속 이어간다
수집 성공: 95092458 [장관동정] 추석 귀성길 안전·편의 빈틈없도록 … 주요 교통현장 집중 점검
수집 성공: 95092447 APEC 교통장관회의서 AI·스마트 모빌리티 미래협력 제안 나선다
수집 성공: 95092445 국토부·기후부, 두 마리 토끼 잡는다... 모두의카드·그린카드 연계
수집 성공: 95092439 추석 연휴 교통혼잡 해소에 총력... 정부 합동 특별교통대책을 시행합니다
수집 성공: 95092438 안전운전 배달 종사자 보험료   최대 11% 할인
수집 성공: 95092430 5극3특 국가균형발전 뒷받침할 초광역교통 전략 연내 마련

===== 신규 수집 결과 =====
수집 성공: 9
수집 실패: 0


,보도자료ID,제목,페이지수,본문글자수
0,95092504,모두의카드 카드정보 현행화하지 않은 이용자는 10월 26일까지 현행화하세요!,2,1622
1,95092486,"택시업종 노동환경 실태, 노 · 사 · 정이 함께 살핀다",2,1205
2,95092459,모두의카드 출퇴근 시차시간 혜택 계속 이어간다,3,2515
3,95092458,[장관동정] 추석 귀성길 안전·편의 빈틈없도록 … 주요 교통현장 집중 점검,2,1774
4,95092447,APEC 교통장관회의서 AI·스마트 모빌리티 미래협력 제안 나선다,5,4158
5,95092445,"국토부·기후부, 두 마리 토끼 잡는다... 모두의카드·그린카드 연계",3,2009
6,95092439,추석 연휴 교통혼잡 해소에 총력... 정부 합동 특별교통대책을 시행합니다,61,48092
7,95092438,안전운전 배달 종사자 보험료 최대 11% 할인,3,2047
8,95092430,5극3특 국가균형발전 뒷받침할 초광역교통 전략 연내 마련,2,1497


In [64]:

from pathlib import Path
import pandas as pd

# 기존 10건의 원본 데이터
existing_df = pd.read_csv("trafficnews_all_10_raw.csv")

# 새로 수집한 9건의 전체 PDF 본문
new_df = page2_extracted_df.copy()

# 기존 데이터와 신규 데이터 통합
combined_df = pd.concat(
    [existing_df, new_df],
    ignore_index=True
)

# 보도자료 ID 형식 통일
combined_df["보도자료ID"] = combined_df["보도자료ID"].astype(str)

# 전체 본문을 기준으로 글자 수 재계산
combined_df["본문글자수"] = combined_df["본문"].str.len()

# 검증
duplicate_count = combined_df["보도자료ID"].duplicated().sum()
missing_count = combined_df["본문"].isna().sum()
empty_count = combined_df["본문"].fillna("").str.strip().eq("").sum()

print("===== 19건 통합 검증 =====")
print("전체 건수:", len(combined_df))
print("중복 ID:", duplicate_count)
print("본문 누락:", missing_count)
print("빈 본문:", empty_count)

# 검증에 실패하면 저장하지 않음
if (
    len(combined_df) != 19
    or duplicate_count != 0
    or missing_count != 0
    or empty_count != 0
):
    raise ValueError("데이터 검증 실패: 저장하지 않습니다.")

# 새 원본 통합 CSV 생성
output_file = Path("trafficnews_all_19_raw.csv")

if output_file.exists():
    print("이미 파일이 존재합니다. 덮어쓰지 않았습니다.")
else:
    combined_df.to_csv(
        output_file,
        index=False,
        encoding="utf-8-sig"
    )
    print("CSV 저장 완료!")

# 저장 후 전체 본문 일치 검사
saved_df = pd.read_csv(output_file)

print("저장 건수:", len(saved_df))
print(
    "저장 전후 본문 일치:",
    saved_df["본문"].tolist() == combined_df["본문"].tolist()
)
print("저장 파일:", output_file)


===== 19건 통합 검증 =====
전체 건수: 19
중복 ID: 0
본문 누락: 0
빈 본문: 0
CSV 저장 완료!
저장 건수: 19
저장 전후 본문 일치: True
저장 파일: trafficnews_all_19_raw.csv


In [66]:

import time
import requests
import pandas as pd
from pathlib import Path
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, parse_qs

# 기존에 수집한 보도자료 ID
existing_df = pd.read_csv("trafficnews_all_19_raw.csv")

existing_ids = set(
    existing_df["보도자료ID"].astype(str)
)

# 신규 보도자료 후보
candidate_records = []
failed_pages = []

base_url = "https://www.molit.go.kr/USR/NEWS/m_71/lst.jsp"

# 3~12페이지 수집
for page_number in range(3, 13):
    page_url = (
        f"{base_url}?lcmspage={page_number}"
        "&search_section=p_sec_5"
    )

    try:
        response = requests.get(page_url, timeout=20)
        response.raise_for_status()
        response.encoding = response.apparent_encoding

        soup = BeautifulSoup(response.text, "html.parser")

        page_count = 0

        for link in soup.select("a[href]"):
            href = link.get("href", "")
            title = link.get_text(" ", strip=True)

            if "dtl.jsp" not in href or not title:
                continue

            detail_url = urljoin(page_url, href)
            query = parse_qs(urlparse(detail_url).query)

            news_id = query.get("id", [""])[0]

            if not news_id or news_id in existing_ids:
                continue

            candidate_records.append({
                "보도자료ID": news_id,
                "제목": title,
                "상세URL": detail_url,
                "목록페이지": page_number,
            })

            page_count += 1

        print(f"{page_number}페이지: 신규 후보 {page_count}건")

    except requests.RequestException as error:
        failed_pages.append(page_number)
        print(f"{page_number}페이지 오류: {error}")

    # 서버 요청 사이에 1초 대기
    time.sleep(1)

# 데이터프레임 생성
candidate_df = pd.DataFrame(
    candidate_records,
    columns=["보도자료ID", "제목", "상세URL", "목록페이지"]
)

# 페이지 사이 중복 제거
candidate_df = candidate_df.drop_duplicates(
    subset=["보도자료ID"]
).reset_index(drop=True)

print("\n===== 수집 결과 =====")
print("신규 후보:", len(candidate_df))
print("기존 원본:", len(existing_df))
print("후보 포함 전체:", len(candidate_df) + len(existing_df))
print("실패 페이지:", failed_pages)

# 기존 파일 덮어쓰기 방지
output_file = Path("trafficnews_candidates.csv")

if output_file.exists():
    print("기존 후보 파일이 있어 덮어쓰지 않았습니다.")
else:
    candidate_df.to_csv(
        output_file,
        index=False,
        encoding="utf-8-sig"
    )
    print("후보 목록 CSV 저장 완료!")

display(candidate_df)


3페이지: 신규 후보 10건
4페이지: 신규 후보 10건
5페이지: 신규 후보 10건
6페이지: 신규 후보 10건
7페이지: 신규 후보 10건
8페이지: 신규 후보 10건
9페이지: 신규 후보 10건
10페이지: 신규 후보 10건
11페이지: 신규 후보 9건
12페이지: 신규 후보 0건

===== 수집 결과 =====
신규 후보: 89
기존 원본: 19
후보 포함 전체: 108
실패 페이지: []
후보 목록 CSV 저장 완료!


,보도자료ID,제목,상세URL,목록페이지
0,95092429,"강릉 찾는 글로벌 방문객, 철도·버스·택시 예약·결제 앱 하나로",https://www.molit.go.kr/USR/NEWS/m_71/dtl.jsp?...,3
1,95092403,AI로 이끄는 환승센터의 미래「제4차 광역교통 환승혁신 포럼」 개최,https://www.molit.go.kr/USR/NEWS/m_71/dtl.jsp?...,3
2,95092377,추석 명절 택배 특별관리기간 4주간 운영,https://www.molit.go.kr/USR/NEWS/m_71/dtl.jsp?...,3
3,95092375,"[장관동정] 김윤덕 장관, SNS로 찾은 “따뜻한 버스기사” 직접 만나 표창",https://www.molit.go.kr/USR/NEWS/m_71/dtl.jsp?...,3
4,95092374,"김용석 대광위원장, 전주 기린대로 BRT 개통 준비상황 현장 점검",https://www.molit.go.kr/USR/NEWS/m_71/dtl.jsp?...,3
...,...,...,...,...
84,95091441,"대광위, ‘K-도시철도’ 원 팀으로 베트남 협력의 새 문을 열다",https://www.molit.go.kr/USR/NEWS/m_71/dtl.jsp?...,11
85,95091427,"공공기관 유휴부지, 생활물류시설로 활용",https://www.molit.go.kr/USR/NEWS/m_71/dtl.jsp?...,11
86,95091425,"신도시 교통대책 TF, 현장 협업 강화로 갈등사업 조정 성과",https://www.molit.go.kr/USR/NEWS/m_71/dtl.jsp?...,11
87,95091414,"[장관동정] 김윤덕 장관, “교통산업이 경쟁력을 높일 수 있도록 교통산업 혁신을 뒷...",https://www.molit.go.kr/USR/NEWS/m_71/dtl.jsp?...,11


In [67]:

import pandas as pd

# 앞서 수집한 후보 89건 불러오기
df = pd.read_csv("trafficnews_candidates.csv")

# 교통정보 관련 키워드
keywords = [
    "교통", "도로", "철도", "버스", "택시",
    "환승", "혼잡", "정체", "고속", "통행",
    "모빌리티", "BRT", "광역", "물류", "운송",
    "안전운전", "자율주행", "대중교통"
]

# 제목에 키워드가 있는지 확인
pattern = "|".join(keywords)

df["관련후보"] = df["제목"].str.contains(
    pattern, case=False, na=False, regex=True
)

print("전체 후보:", len(df))
print("키워드 일치:", df["관련후보"].sum())
print("추가 검토:", (~df["관련후보"]).sum())

# 전체 89건을 분류 결과와 함께 표시
with pd.option_context("display.max_rows", None):
    display(df[["보도자료ID", "제목", "관련후보"]])


전체 후보: 89
키워드 일치: 62
추가 검토: 27


,보도자료ID,제목,관련후보
0,95092429,"강릉 찾는 글로벌 방문객, 철도·버스·택시 예약·결제 앱 하나로",True
1,95092403,AI로 이끄는 환승센터의 미래「제4차 광역교통 환승혁신 포럼」 개최,True
2,95092377,추석 명절 택배 특별관리기간 4주간 운영,False
3,95092375,"[장관동정] 김윤덕 장관, SNS로 찾은 “따뜻한 버스기사” 직접 만나 표창",True
4,95092374,"김용석 대광위원장, 전주 기린대로 BRT 개통 준비상황 현장 점검",True
5,95092358,환승센터 구축으로 전국을 더 편리하고 가깝게,True
6,95092355,위례선 트램 건설사업 갈등조정안 심의·의결,False
7,95092342,‘꼼수 환급’ 막는다 모두의카드 부정수급 관리 강화,False
8,95092266,모두의카드(기후동행패스) 서비스 9월 1일부터 시행,False
9,95092246,올여름 안전한 휴가길 최우선… 극한호우·기상이변 철저 대응,False


In [69]:
import pandas as pd
import re

# 기존 후보와 분류 결과 사용
review_candidates = df.copy()

# 교통정보 관련 추가 키워드
extra_keywords = [
    "트램", "노면전차", "지하철", "전철",
    "모두의카드", "K-패스", "카드",
    "유가보조금", "주차", "안전운임",
    "택배", "배달", "이동", "출퇴근",
    "교통난", "연장", "차량", "공항",
    "운임", "환급", "운전자"
]

pattern = "|".join(re.escape(word) for word in extra_keywords)

# 기존 True는 유지하고 False인 자료만 추가 검사
review_candidates["추가키워드일치"] = (
    review_candidates["제목"].str.contains(
        pattern, case=False, na=False
    )
)

review_candidates["분류"] = "추가 검토"

review_candidates.loc[
    review_candidates["관련후보"]
    | review_candidates["추가키워드일치"],
    "분류"
] = "관련 후보"

print("전체:", len(review_candidates))
print(review_candidates["분류"].value_counts())

display(
    review_candidates[
        ["보도자료ID", "제목", "분류"]
    ]
)

전체: 89
분류
관련 후보    86
추가 검토     3
Name: count, dtype: int64


,보도자료ID,제목,분류
0,95092429,"강릉 찾는 글로벌 방문객, 철도·버스·택시 예약·결제 앱 하나로",관련 후보
1,95092403,AI로 이끄는 환승센터의 미래「제4차 광역교통 환승혁신 포럼」 개최,관련 후보
2,95092377,추석 명절 택배 특별관리기간 4주간 운영,관련 후보
3,95092375,"[장관동정] 김윤덕 장관, SNS로 찾은 “따뜻한 버스기사” 직접 만나 표창",관련 후보
4,95092374,"김용석 대광위원장, 전주 기린대로 BRT 개통 준비상황 현장 점검",관련 후보
...,...,...,...
84,95091441,"대광위, ‘K-도시철도’ 원 팀으로 베트남 협력의 새 문을 열다",관련 후보
85,95091427,"공공기관 유휴부지, 생활물류시설로 활용",관련 후보
86,95091425,"신도시 교통대책 TF, 현장 협업 강화로 갈등사업 조정 성과",관련 후보
87,95091414,"[장관동정] 김윤덕 장관, “교통산업이 경쟁력을 높일 수 있도록 교통산업 혁신을 뒷...",관련 후보


In [70]:

import csv
import time
import requests
import pymupdf
import pandas as pd
from pathlib import Path
from bs4 import BeautifulSoup
from urllib.parse import urljoin

# 수집 후보 89건
candidates = pd.read_csv("trafficnews_candidates.csv")
output_file = Path("trafficnews_candidates_raw.csv")

columns = [
    "보도자료ID", "제목", "담당부서", "등록일",
    "상세URL", "PDF_URL", "본문", "페이지수", "본문글자수"
]

# 이전 실행에서 저장된 ID 확인
if output_file.exists():
    completed_ids = set(
        pd.read_csv(output_file)["보도자료ID"].astype(str)
    )
else:
    completed_ids = set()

failures = []

with requests.Session() as session:
    for _, row in candidates.iterrows():
        news_id = str(row["보도자료ID"])

        if news_id in completed_ids:
            continue

        try:
            # 상세 페이지 요청
            response = session.get(row["상세URL"], timeout=30)
            response.raise_for_status()
            response.encoding = response.apparent_encoding

            soup = BeautifulSoup(response.text, "html.parser")

            title_tag = soup.select_one("h4")
            if title_tag is None:
                raise ValueError("제목 추출 실패")

            title = title_tag.get_text(" ", strip=True)
            department = ""
            published_at = ""

            for item in soup.select("ul.bd_view_ul_info li"):
                label = item.select_one("strong")
                value = item.select_one("span")

                if label is None or value is None:
                    continue

                key = label.get_text(strip=True)
                text = value.get_text(" ", strip=True)

                if key == "담당부서":
                    department = text
                elif key == "등록일":
                    published_at = text

            pdf_url = ""

            for link in soup.select("li.file a[href]"):
                name = link.get_text(" ", strip=True)

                if ".pdf" in name.lower():
                    pdf_url = urljoin(
                        row["상세URL"], link["href"]
                    )
                    break

            if not pdf_url:
                raise ValueError("PDF 첨부파일 없음")

            time.sleep(1)

            # PDF 다운로드 및 전체 본문 추출
            pdf_response = session.get(pdf_url, timeout=60)
            pdf_response.raise_for_status()

            if not pdf_response.content.startswith(b"%PDF-"):
                raise ValueError("PDF 파일 형식 오류")

            with pymupdf.open(
                stream=pdf_response.content,
                filetype="pdf"
            ) as document:
                page_count = len(document)
                full_text = "\n".join(
                    page.get_text("text")
                    for page in document
                )

            if not full_text.strip():
                raise ValueError("추출 본문이 비어 있음")

            record = {
                "보도자료ID": news_id,
                "제목": title,
                "담당부서": department,
                "등록일": published_at,
                "상세URL": row["상세URL"],
                "PDF_URL": pdf_url,
                "본문": full_text,
                "페이지수": page_count,
                "본문글자수": len(full_text),
            }

            # 성공한 자료만 CSV에 추가
            is_new_file = not output_file.exists()

            with output_file.open(
                "a", newline="", encoding="utf-8-sig"
            ) as file:
                writer = csv.DictWriter(file, fieldnames=columns)

                if is_new_file:
                    writer.writeheader()

                writer.writerow(record)

            completed_ids.add(news_id)
            print(f"성공: {news_id} / {len(full_text)}자")

        except (
            requests.RequestException,
            pymupdf.FileDataError,
            ValueError,
            OSError
        ) as error:
            failures.append({
                "보도자료ID": news_id,
                "오류": str(error)
            })
            print(f"실패: {news_id} / {error}")

        # 다음 자료 요청 전 대기
        time.sleep(1)

print("\n===== 수집 결과 =====")

if output_file.exists():
    saved = pd.read_csv(output_file)

    print("수집 완료:", len(saved))
    print("중복 ID:", saved["보도자료ID"].duplicated().sum())
    print("빈 본문:", saved["본문"].fillna("").str.strip().eq("").sum())
    print("저장 파일:", output_file)
else:
    print("저장된 자료가 없습니다.")

print("이번 실행 실패:", len(failures))

if failures:
    display(pd.DataFrame(failures))


성공: 95092429 / 1714자
성공: 95092403 / 3149자
성공: 95092377 / 1000자
성공: 95092375 / 901자
성공: 95092374 / 1183자
성공: 95092358 / 5709자
성공: 95092355 / 1941자
성공: 95092342 / 1263자
성공: 95092266 / 3808자
성공: 95092246 / 42607자
성공: 95092241 / 2090자
성공: 95092235 / 1766자
성공: 95092229 / 1414자
성공: 95092218 / 3241자
성공: 95092207 / 913자
성공: 95092194 / 3361자
성공: 95092191 / 1946자
성공: 95092189 / 1921자
성공: 95092184 / 1064자
성공: 95092172 / 887자
성공: 95092154 / 6580자
성공: 95092145 / 2016자
성공: 95092135 / 1277자
성공: 95092132 / 1065자
성공: 95092130 / 1215자
성공: 95092124 / 3218자
성공: 95092120 / 1552자
성공: 95092113 / 1165자
성공: 95092102 / 1177자
성공: 95092091 / 1964자
성공: 95092072 / 1862자
성공: 95092068 / 1188자
성공: 95092053 / 3357자
성공: 95092050 / 1510자
성공: 95092048 / 1287자
성공: 95092032 / 1001자
성공: 95092022 / 3747자
성공: 95091996 / 1884자
성공: 95091987 / 1429자
성공: 95091971 / 1804자
성공: 95091964 / 1301자
성공: 95091931 / 3004자
성공: 95091920 / 1076자
MuPDF error: syntax error: invalid key in dict

MuPDF error: syntax error: invalid key in dict

성공:

In [71]:

import csv
import time
import requests
import pymupdf
import pandas as pd
from pathlib import Path
from bs4 import BeautifulSoup
from urllib.parse import urljoin

# 수집 후보 89건
candidates = pd.read_csv("trafficnews_candidates.csv")
output_file = Path("trafficnews_candidates_raw.csv")

columns = [
    "보도자료ID", "제목", "담당부서", "등록일",
    "상세URL", "PDF_URL", "본문", "페이지수", "본문글자수"
]

# 이전 실행에서 저장된 ID 확인
if output_file.exists():
    completed_ids = set(
        pd.read_csv(output_file)["보도자료ID"].astype(str)
    )
else:
    completed_ids = set()

failures = []

with requests.Session() as session:
    for _, row in candidates.iterrows():
        news_id = str(row["보도자료ID"])

        if news_id in completed_ids:
            continue

        try:
            # 상세 페이지 요청
            response = session.get(row["상세URL"], timeout=30)
            response.raise_for_status()
            response.encoding = response.apparent_encoding

            soup = BeautifulSoup(response.text, "html.parser")

            title_tag = soup.select_one("h4")
            if title_tag is None:
                raise ValueError("제목 추출 실패")

            title = title_tag.get_text(" ", strip=True)
            department = ""
            published_at = ""

            for item in soup.select("ul.bd_view_ul_info li"):
                label = item.select_one("strong")
                value = item.select_one("span")

                if label is None or value is None:
                    continue

                key = label.get_text(strip=True)
                text = value.get_text(" ", strip=True)

                if key == "담당부서":
                    department = text
                elif key == "등록일":
                    published_at = text

            pdf_url = ""

            for link in soup.select("li.file a[href]"):
                name = link.get_text(" ", strip=True)

                if ".pdf" in name.lower():
                    pdf_url = urljoin(
                        row["상세URL"], link["href"]
                    )
                    break

            if not pdf_url:
                raise ValueError("PDF 첨부파일 없음")

            time.sleep(1)

            # PDF 다운로드 및 전체 본문 추출
            pdf_response = session.get(pdf_url, timeout=60)
            pdf_response.raise_for_status()

            if not pdf_response.content.startswith(b"%PDF-"):
                raise ValueError("PDF 파일 형식 오류")

            with pymupdf.open(
                stream=pdf_response.content,
                filetype="pdf"
            ) as document:
                page_count = len(document)
                full_text = "\n".join(
                    page.get_text("text")
                    for page in document
                )

            if not full_text.strip():
                raise ValueError("추출 본문이 비어 있음")

            record = {
                "보도자료ID": news_id,
                "제목": title,
                "담당부서": department,
                "등록일": published_at,
                "상세URL": row["상세URL"],
                "PDF_URL": pdf_url,
                "본문": full_text,
                "페이지수": page_count,
                "본문글자수": len(full_text),
            }

            # 성공한 자료만 CSV에 추가
            is_new_file = not output_file.exists()

            with output_file.open(
                "a", newline="", encoding="utf-8-sig"
            ) as file:
                writer = csv.DictWriter(file, fieldnames=columns)

                if is_new_file:
                    writer.writeheader()

                writer.writerow(record)

            completed_ids.add(news_id)
            print(f"성공: {news_id} / {len(full_text)}자")

        except (
            requests.RequestException,
            pymupdf.FileDataError,
            ValueError,
            OSError
        ) as error:
            failures.append({
                "보도자료ID": news_id,
                "오류": str(error)
            })
            print(f"실패: {news_id} / {error}")

        # 다음 자료 요청 전 대기
        time.sleep(1)

print("\n===== 수집 결과 =====")

if output_file.exists():
    saved = pd.read_csv(output_file)

    print("수집 완료:", len(saved))
    print("중복 ID:", saved["보도자료ID"].duplicated().sum())
    print("빈 본문:", saved["본문"].fillna("").str.strip().eq("").sum())
    print("저장 파일:", output_file)
else:
    print("저장된 자료가 없습니다.")

print("이번 실행 실패:", len(failures))

if failures:
    display(pd.DataFrame(failures))



===== 수집 결과 =====
수집 완료: 89
중복 ID: 0
빈 본문: 0
저장 파일: trafficnews_candidates_raw.csv
이번 실행 실패: 0


In [72]:
from pathlib import Path
import pandas as pd

old_df = pd.read_csv("trafficnews_all_19_raw.csv")
new_df = pd.read_csv("trafficnews_candidates_raw.csv")

# 전체 원본 통합
combined_df = pd.concat([old_df, new_df], ignore_index=True)

# 보도자료 ID 기준 검증
duplicate_count = combined_df["보도자료ID"].duplicated().sum()
missing_count = combined_df["본문"].isna().sum()
empty_count = combined_df["본문"].fillna("").str.strip().eq("").sum()

print("전체 건수:", len(combined_df))
print("중복 ID:", duplicate_count)
print("본문 누락:", missing_count)
print("빈 본문:", empty_count)

if (
    len(combined_df) != 108
    or duplicate_count != 0
    or missing_count != 0
    or empty_count != 0
):
    raise ValueError("통합 데이터 검증 실패")

output_file = Path("trafficnews_all_108_raw.csv")

if output_file.exists():
    print("기존 파일이 있어 덮어쓰지 않았습니다.")
else:
    combined_df.to_csv(
        output_file,
        index=False,
        encoding="utf-8-sig"
    )
    print("108건 통합 CSV 저장 완료!")

# 저장 후 전체 본문 일치 검사
saved_df = pd.read_csv(output_file)

print("저장 건수:", len(saved_df))
print(
    "본문 전체 일치:",
    saved_df["본문"].tolist() == combined_df["본문"].tolist()
)

전체 건수: 108
중복 ID: 0
본문 누락: 0
빈 본문: 0
108건 통합 CSV 저장 완료!
저장 건수: 108
본문 전체 일치: True
